In [ ]:
# Cell 1 — Objective: Install only the required packages for this notebook; import nothing sensitive here.

%pip install --quiet --upgrade pip
%pip install --quiet oci ipywidgets paramiko tqdm numpy pandas locust requests

print("Cell 1 complete: Python environment ready.")
print("NEXT: Run Cell 2 to load central variables and validate OCI config (CPU/Throughput knobs included).")


In [ ]:
# Cell 2 — Objective: Central variables and credentials (single source of truth; no network calls)
# Includes CPU worker policy for generators, throughput payload controls, and backend target selection.

import os
from datetime import datetime, timezone
import oci

# OCI config (no secrets hardcoded; load from standard ~/.oci/config)
OCI_CONFIG_FILE = os.path.expanduser(os.environ.get("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE = os.environ.get("OCI_PROFILE", "DEFAULT")

_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
oci.config.validate_config(_cfg)

TENANCY_OCID = os.environ.get("TENANCY_OCID", _cfg["tenancy"])
USER_OCID = os.environ.get("USER_OCID", _cfg["user"])
FINGERPRINT = os.environ.get("FINGERPRINT", _cfg["fingerprint"])
OCI_PRIVATE_KEY_PATH = os.path.expanduser(
    os.environ.get("OCI_PRIVATE_KEY_PATH", _cfg.get("key_file", ""))
)
PRIVATE_KEY_PASSPHRASE = os.environ.get(
    "OCI_PASSPHRASE",
    os.environ.get("OCI_PRIVATE_KEY_PASSPHRASE", _cfg.get("pass_phrase", "")),
)
REGION = os.environ.get("REGION", _cfg["region"])

# SSH keys defaults (can be changed in Cell 3)
_default_ssh_pub = os.path.expanduser(
    os.environ.get("SSH_PUBLIC_KEY_PATH", "~/.ssh/id_rsa.pub")
)
_default_ssh_priv = os.path.expanduser(
    os.environ.get("SSH_PRIVATE_KEY_PATH", "~/.ssh/id_rsa")
)
SSH_PUBLIC_KEY_PATH = _default_ssh_pub if os.path.exists(_default_ssh_pub) else ""
SSH_PRIVATE_KEY_PATH = _default_ssh_priv if os.path.exists(_default_ssh_priv) else ""

# Topology defaults (adjustable in Cell 3)
BACKEND_COUNT = int(os.environ.get("BACKEND_COUNT", "4"))
GENERATOR_COUNT = int(os.environ.get("GENERATOR_COUNT", "4"))

BACKEND_SHAPE = os.environ.get("BACKEND_SHAPE", "VM.Standard.E5.Flex")
BACKEND_OCPUS = float(os.environ.get("BACKEND_OCPUS", "16"))
BACKEND_MEMORY_GB = float(os.environ.get("BACKEND_MEMORY_GB", "64"))

GENERATOR_SHAPE = os.environ.get("GENERATOR_SHAPE", "VM.Standard.E5.Flex")
GENERATOR_OCPUS = float(os.environ.get("GENERATOR_OCPUS", "16"))
GENERATOR_MEMORY_GB = float(os.environ.get("GENERATOR_MEMORY_GB", "64"))

# TLS protocol (used by backends; supports TLS 1.2/1.3)
TLS_PROTOCOL = os.environ.get("TLS_PROTOCOL", "TLSv1.3")

# Backends endpoints
HEALTH_ENDPOINT_PATH = os.environ.get("HEALTH_ENDPOINT_PATH", "/healthz")
THROUGHPUT_ENDPOINT_PATH = os.environ.get(
    "THROUGHPUT_ENDPOINT_PATH", "/payload_100k"
)  # overwritten after Apply in Cell 3

# Locust client behavior
LOCUST_WAIT_TIME_SEC = float(os.environ.get("LOCUST_WAIT_TIME_SEC", "1.0"))
LOCUST_CONNECT_TIMEOUT = int(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000"))  # ms
LOCUST_READ_TIMEOUT = int(os.environ.get("LOCUST_READ_TIMEOUT_MS", "15000"))  # ms
LOCUST_VERIFY_TLS = (
    os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
)  # False for self-signed

# Remote workspace (must NOT be named "locust" to avoid Python package shadowing)
LOCUST_WORKDIR = os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork")

# Worker wait policy (avoid stalls by default)
EXPECT_WORKERS_STRICT = (
    os.environ.get("EXPECT_WORKERS_STRICT", "false").lower() == "true"
)
EXPECTED_WORKERS_OVERRIDE = (
    None
    if os.environ.get("EXPECTED_WORKERS_OVERRIDE", "").strip() == ""
    else int(os.environ.get("EXPECTED_WORKERS_OVERRIDE"))
)

# Poll grace period after hold (seconds)
GRACE_SEC = int(os.environ.get("GRACE_SEC", "60"))

# Locust UI toggles
UI_ENABLE = True
UI_EXPOSE_MODE = "tunnel"  # "tunnel" or "nsg"
UI_WEB_HOST = "0.0.0.0"
UI_WEB_PORT = int(os.environ.get("UI_WEB_PORT", "8089"))
UI_ALLOWED_CIDR = os.environ.get("UI_ALLOWED_CIDR", "0.0.0.0/0")  # if using NSG

# Test Mode selector (default CPS)
TEST_MODE = os.environ.get("TEST_MODE", "cps").lower()  # "cps" or "throughput"

# CPS tiers & durations (adjustable in Cell 3)
CPS_TIERS = [10000, 25000, 35000, 50000, 100000]
CPS_WARMUPS = {10000: 120, 25000: 150, 35000: 160, 50000: 180, 100000: 210}
CPS_HOLDS = {10000: 600, 25000: 600, 35000: 600, 50000: 600, 100000: 600}

# Throughput configuration (adjustable in Cell 3)
TPUT_TARGETS_GBPS_TEXT = "1,5,10"
TPUT_WARMUP_SEC = 120
TPUT_HOLD_SEC = 600

# Payload knobs (arbitrary sizes supported, e.g., 4k, 5k, 10k, 100k, 1m)
TPUT_PAYLOAD_SIZE_TEXT = "100k"  # selected run-time payload
TPUT_PAYLOAD_SIZES_TEXT = "4k,5k,10k,50k,100k,256k,1m,5m"  # baked on backends

# Derived at apply time (Cell 3)
TPUT_PAYLOAD_SIZE_BYTES = 100_000
TPUT_PAYLOAD_SIZE_LABEL = "100k"
TPUT_PAYLOAD_BYTES_PER_REQ = TPUT_PAYLOAD_SIZE_BYTES

# Generator CPU → worker policy (adjustable in Cell 3)
WORKERS_PER_HOST = "auto"  # "auto" or an integer
CPU_RESERVE = 1  # leave 1 for host
MIN_WORKERS_PER_HOST = 1
MAX_WORKERS_PER_HOST = 32

# Master private IP override (optional)
MASTER_PRIVATE_IP_OVERRIDE = os.environ.get("MASTER_PRIVATE_IP_OVERRIDE", "").strip()

# Access control for generator SSH (public IPs for ease, configurable)
SSH_ALLOWED_CIDR = os.environ.get("SSH_ALLOWED_CIDR", "0.0.0.0/0")

# Backend target selection (index into BACKEND_IPS after provisioning)
BACKEND_TARGET_INDEX = int(os.environ.get("BACKEND_TARGET_INDEX", "0"))

# Output directory and UTC timestamp
OUTPUT_DIR = os.path.abspath(os.environ.get("OUTPUT_DIR", "./results"))
os.makedirs(OUTPUT_DIR, exist_ok=True)
TS_UTC = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# Placeholders (populated in Cell 3 and later)
SELECTED_REGION = None
COMPARTMENT_ID = ""
AD_A = ""
IMAGE_ID = ""
SSH_PUBLIC_KEY_CONTENT = ""


def _ensure_path_exists(path: str, label: str):
    if path and not os.path.exists(os.path.expanduser(path)):
        raise FileNotFoundError(f"{label} not found: {path}")


_ensure_path_exists(OCI_PRIVATE_KEY_PATH, "OCI private key")

if not SSH_PUBLIC_KEY_PATH:
    print(
        "Note: SSH public key not found at default path; you will select a key in Cell 3."
    )
if not SSH_PRIVATE_KEY_PATH:
    print(
        "Note: SSH private key not found at default path; you will select a key in Cell 3."
    )

print(
    f"Cell 2 complete: Config loaded from {OCI_CONFIG_FILE} [{OCI_PROFILE}] for tenancy: {TENANCY_OCID}"
)
print(
    "NEXT: Run Cell 3, select shapes, keys, test mode/payload/CPU worker policy, and click 'Apply Selections'."
)

In [ ]:
# Cell 3 — Objective: Interactive dashboard: Region/Compartment/AD/Shapes/Keys + Test Mode, Payload, CPU policy (CPS or Throughput).

import os
from pathlib import Path
import oci
from IPython.display import display, HTML
import ipywidgets as widgets

SECTION_BG = "#f8f9fb"
BORDER = "1px solid #e0e0e0"
PAD = "12px"
CONTAINER_MAX_W = "96%"


def section(title_text, body_widgets):
    title = widgets.HTML(f"<b>{title_text}</b>")
    body = (
        body_widgets
        if isinstance(body_widgets, widgets.Widget)
        else widgets.VBox(body_widgets)
    )
    return widgets.VBox(
        [title, body],
        layout=widgets.Layout(
            width="100%",
            border=BORDER,
            padding=PAD,
            margin="8px 0",
            background_color=SECTION_BG,
        ),
    )


def row(*children, gap="12px"):
    return widgets.HBox(list(children), layout=widgets.Layout(gap=gap, width="100%"))


def vspace(h="6px"):
    return widgets.HTML(f"<div style='height:{h}'></div>")


def build_signer(tenancy, user, fp, key_path, passphrase, cfg):
    return oci.signer.Signer(
        tenancy=tenancy,
        user=user,
        fingerprint=fp,
        private_key_file_location=key_path,
        pass_phrase=passphrase if passphrase else None,
        private_key_content=cfg.get("key_content"),
    )


def cfg_for_region(region_name):
    c = dict(_cfg)
    c["region"] = region_name
    return c


_base_signer = build_signer(
    TENANCY_OCID,
    USER_OCID,
    FINGERPRINT,
    OCI_PRIVATE_KEY_PATH,
    PRIVATE_KEY_PASSPHRASE,
    _cfg,
)
idc_base = oci.identity.IdentityClient(config=_cfg, signer=_base_signer)
subs = sorted(
    idc_base.list_region_subscriptions(TENANCY_OCID).data, key=lambda r: r.region_name
)
region_options = [(r.region_name, r.region_name) for r in subs]
default_region = (
    REGION
    if REGION in [r.region_name for r in subs]
    else (next((r.region_name for r in subs if r.is_home_region), subs[0].region_name))
)

region_dd = widgets.Dropdown(
    options=region_options,
    value=default_region,
    description="Region:",
    layout=widgets.Layout(width="100%"),
)
reload_btn = widgets.Button(description="Reload", icon="refresh")
reset_btn = widgets.Button(description="Reset", icon="history")
err_out, summary_out = widgets.Output(), widgets.Output()
dynamic_box = widgets.VBox([])


def discover_files(dirs, exts=None, include_hidden=True):
    out = []
    for d in dirs:
        p = Path(os.path.expanduser(d))
        if not p.exists() or not p.is_dir():
            continue
        for f in p.iterdir():
            if not f.is_file():
                continue
            if not include_hidden and f.name.startswith("."):
                continue
            if exts is not None and not any(str(f).endswith(ext) for ext in exts):
                continue
            out.append(str(f))
    out.sort(
        key=lambda s: Path(s).stat().st_mtime if Path(s).exists() else 0, reverse=True
    )
    return out


ssh_dir = os.path.expanduser("~/.ssh")
ssh_pub_candidates = [p for p in discover_files([ssh_dir], exts=[".pub"])]
ssh_priv_candidates = [
    p for p in discover_files([ssh_dir], exts=None) if not p.endswith(".pub")
]
ssh_pub_default = next(
    (p for p in ssh_pub_candidates if p == SSH_PUBLIC_KEY_PATH),
    (ssh_pub_candidates[0] if ssh_pub_candidates else ""),
)
ssh_priv_default = next(
    (p for p in ssh_priv_candidates if p == SSH_PRIVATE_KEY_PATH),
    (ssh_priv_candidates[0] if ssh_priv_candidates else ""),
)

ssh_pub_dd = widgets.Dropdown(
    options=[(p, p) for p in ssh_pub_candidates]
    or [("No *.pub keys found in ~/.ssh", "")],
    value=ssh_pub_default,
    description="SSH pub:",
    layout=widgets.Layout(width="100%"),
)
ssh_priv_dd = widgets.Dropdown(
    options=[(p, p) for p in ssh_priv_candidates]
    or [("No private keys found in ~/.ssh", "")],
    value=ssh_priv_default,
    description="SSH priv:",
    layout=widgets.Layout(width="100%"),
)

# Locust/Test params
tls_dd = widgets.Dropdown(
    options=["TLSv1.3", "TLSv1.2"], value=TLS_PROTOCOL, description="TLS Protocol:"
)
health_ep_in = widgets.Text(
    value=HEALTH_ENDPOINT_PATH,
    description="Health Path:",
    layout=widgets.Layout(width="100%"),
)
throughput_ep_in = widgets.Text(
    value=THROUGHPUT_ENDPOINT_PATH,
    description="Throughput Base Path:",
    layout=widgets.Layout(width="100%"),
)
ssh_cidr_in = widgets.Text(
    value=SSH_ALLOWED_CIDR,
    description="SSH CIDR (generators):",
    layout=widgets.Layout(width="100%"),
)

test_mode_dd = widgets.Dropdown(
    options=[
        ("CPS (connections/sec)", "cps"),
        ("Throughput (Gbps via payload)", "throughput"),
    ],
    value=TEST_MODE,
    description="Test Mode:",
)
wait_time_in = widgets.FloatText(
    value=LOCUST_WAIT_TIME_SEC, description="Wait(s)/user:", step=0.1
)
conn_timeout_in = widgets.BoundedIntText(
    value=LOCUST_CONNECT_TIMEOUT,
    min=1000,
    max=60000,
    step=500,
    description="Connect ms:",
)
read_timeout_in = widgets.BoundedIntText(
    value=LOCUST_READ_TIMEOUT, min=1000, max=120000, step=500, description="Read ms:"
)
verify_tls_in = widgets.Checkbox(
    value=LOCUST_VERIFY_TLS, description="Verify TLS (False for self-signed)"
)

ui_enable_in = widgets.Checkbox(value=UI_ENABLE, description="Enable Locust UI")
ui_mode_in = widgets.Dropdown(
    options=["tunnel", "nsg"], value=UI_EXPOSE_MODE, description="UI expose mode:"
)
ui_port_in = widgets.BoundedIntText(
    value=UI_WEB_PORT, min=1024, max=65535, step=1, description="UI port:"
)
ui_cidr_in = widgets.Text(value=UI_ALLOWED_CIDR, description="UI allowed CIDR:")


# CPS tiers/durations
def _int_box(v, desc):
    return widgets.BoundedIntText(
        value=int(v), min=1, max=36000, step=1, description=desc
    )


warm_10k_in, hold_10k_in = _int_box(CPS_WARMUPS[10000], "10k warmup(s):"), _int_box(
    CPS_HOLDS[10000], "10k hold(s):"
)
warm_25k_in, hold_25k_in = _int_box(CPS_WARMUPS[25000], "25k warmup(s):"), _int_box(
    CPS_HOLDS[25000], "25k hold(s):"
)
warm_35k_in, hold_35k_in = _int_box(CPS_WARMUPS[35000], "35k warmup(s):"), _int_box(
    CPS_HOLDS[35000], "35k hold(s):"
)
warm_50k_in, hold_50k_in = _int_box(CPS_WARMUPS[50000], "50k warmup(s):"), _int_box(
    CPS_HOLDS[50000], "50k hold(s):"
)
warm_100k_in, hold_100k_in = _int_box(CPS_WARMUPS[100000], "100k warmup(s):"), _int_box(
    CPS_HOLDS[100000], "100k hold(s):"
)

# Throughput targets/durations
tput_targets_in = widgets.Text(
    value=TPUT_TARGETS_GBPS_TEXT,
    description="TPUT targets (Gbps):",
    layout=widgets.Layout(width="100%"),
)
tput_warm_in = widgets.BoundedIntText(
    value=int(TPUT_WARMUP_SEC), min=1, max=36000, step=1, description="TPUT warmup(s):"
)
tput_hold_in = widgets.BoundedIntText(
    value=int(TPUT_HOLD_SEC), min=1, max=36000, step=1, description="TPUT hold(s):"
)
tput_payload_size_in = widgets.Text(
    value=TPUT_PAYLOAD_SIZE_TEXT,
    description="TPUT payload for run (e.g., 4k, 5k, 10k, 100k, 1m):",
)
tput_payload_bake_in = widgets.Text(
    value=TPUT_PAYLOAD_SIZES_TEXT,
    description="Payload sizes to bake (comma-separated):",
    layout=widgets.Layout(width="100%"),
)
payload_help = widgets.HTML(
    "<i>Suffixes: k ≈1000 bytes, m ≈1,000,000 bytes. Examples: 4k, 5k, 10k, 100k, 1m, 5m.</i>"
)

# CPU policy controls
workers_per_host_mode_in = widgets.Dropdown(
    options=[("Auto (use OCPUs)", "auto"), ("Fixed count", "fixed")],
    value="auto",
    description="Workers/host mode:",
)
fixed_workers_in = widgets.BoundedIntText(
    value=16, min=1, max=256, step=1, description="Fixed workers/host:"
)
cpu_reserve_in = widgets.BoundedIntText(
    value=int(CPU_RESERVE), min=0, max=8, step=1, description="CPU reserve:"
)
min_workers_in = widgets.BoundedIntText(
    value=int(MIN_WORKERS_PER_HOST),
    min=1,
    max=256,
    step=1,
    description="Min workers/host:",
)
max_workers_in = widgets.BoundedIntText(
    value=int(MAX_WORKERS_PER_HOST),
    min=1,
    max=512,
    step=1,
    description="Max workers/host:",
)

# Counts and shapes
backend_count_in = widgets.BoundedIntText(
    value=int(BACKEND_COUNT), min=1, max=64, step=1, description="Backends:"
)
generator_count_in = widgets.BoundedIntText(
    value=int(GENERATOR_COUNT), min=1, max=128, step=1, description="Generators:"
)

apply_btn = widgets.Button(
    description="Apply Selections",
    button_style="primary",
    icon="check",
    layout=widgets.Layout(width="240px", height="36px", align_self="center"),
)

_state = {
    "comp_dd": None,
    "ad_a_dd": None,
    "shape_filter": None,
    "backend_shape_dd": None,
    "generator_shape_dd": None,
    "image_dd": None,
    "backend_ocpus_in": None,
    "backend_mem_in": None,
    "generator_ocpus_in": None,
    "generator_mem_in": None,
}


def identity_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    signer = build_signer(
        TENANCY_OCID,
        USER_OCID,
        FINGERPRINT,
        OCI_PRIVATE_KEY_PATH,
        PRIVATE_KEY_PASSPHRASE,
        cfg_r,
    )
    return oci.identity.IdentityClient(config=cfg_r, signer=signer)


def compute_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    signer = build_signer(
        TENANCY_OCID,
        USER_OCID,
        FINGERPRINT,
        OCI_PRIVATE_KEY_PATH,
        PRIVATE_KEY_PASSPHRASE,
        cfg_r,
    )
    return oci.core.ComputeClient(config=cfg_r, signer=signer)


def list_compartments(idc):
    comps = oci.pagination.list_call_get_all_results(
        idc.list_compartments,
        TENANCY_OCID,
        compartment_id_in_subtree=True,
        access_level="ACCESSIBLE",
    ).data
    comps = [c for c in comps if c.lifecycle_state == "ACTIVE"]
    tenancy = idc.get_tenancy(TENANCY_OCID).data
    tenancy_name = getattr(tenancy, "name", "root-tenancy")
    return [(f"{tenancy_name} (root)", TENANCY_OCID)] + sorted(
        [(c.name + f" ({c.description or 'no-desc'})", c.id) for c in comps],
        key=lambda t: t[0].lower(),
    )


def list_ads(idc):
    ads = oci.pagination.list_call_get_all_results(
        idc.list_availability_domains, TENANCY_OCID
    ).data
    return sorted([ad.name for ad in ads]) or ["AD-1"]


def list_shapes(cc):
    shapes = oci.pagination.list_call_get_all_results(cc.list_shapes, TENANCY_OCID).data
    return sorted({s.shape for s in shapes})


def list_images(cc, comp_id: str, b_shape: str, g_shape: str):
    def imgs_for(shape):
        return oci.pagination.list_call_get_all_results(
            cc.list_images,
            comp_id,
            operating_system="Oracle Linux",
            sort_by="TIMECREATED",
            sort_order="DESC",
            shape=shape,
        ).data

    if b_shape and g_shape:
        b, g = imgs_for(b_shape), imgs_for(g_shape)
        g_ids = {im.id for im in g}
        return [im for im in b if im.id in g_ids]
    shape = b_shape or g_shape
    return (
        imgs_for(shape)
        if shape
        else oci.pagination.list_call_get_all_results(
            cc.list_images,
            comp_id,
            operating_system="Oracle Linux",
            sort_by="TIMECREATED",
            sort_order="DESC",
        ).data
    )


def show_flex_inputs(back_shape: str, gen_shape: str):
    if _state["backend_ocpus_in"] is None:
        _state["backend_ocpus_in"] = widgets.BoundedIntText(
            value=int(BACKEND_OCPUS),
            min=1,
            max=128,
            step=1,
            description="Backend OCPUs:",
        )
        _state["backend_mem_in"] = widgets.BoundedIntText(
            value=int(BACKEND_MEMORY_GB),
            min=1,
            max=2048,
            step=1,
            description="Backend Memory(GB):",
        )
        _state["generator_ocpus_in"] = widgets.BoundedIntText(
            value=int(GENERATOR_OCPUS),
            min=1,
            max=256,
            step=1,
            description="Generator OCPUs:",
        )
        _state["generator_mem_in"] = widgets.BoundedIntText(
            value=int(GENERATOR_MEMORY_GB),
            min=1,
            max=4096,
            step=1,
            description="Generator Memory(GB):",
        )
    _state["backend_ocpus_in"].layout.display = (
        "block" if (back_shape or "").endswith(".Flex") else "none"
    )
    _state["backend_mem_in"].layout.display = (
        "block" if (back_shape or "").endswith(".Flex") else "none"
    )
    _state["generator_ocpus_in"].layout.display = (
        "block" if (gen_shape or "").endswith(".Flex") else "none"
    )
    _state["generator_mem_in"].layout.display = (
        "block" if (gen_shape or "").endswith(".Flex") else "none"
    )


def rebuild_dynamic_area(_=None):
    err_out.clear_output()
    with err_out:
        print(f"Refreshing for region: {region_dd.value} ...")
    try:
        idc = identity_client_for_current()
        cc = compute_client_for_current()

        comp_opts = list_compartments(idc)
        comp_dd = widgets.Dropdown(
            options=comp_opts,
            value=comp_opts[0][1],
            description="Compartment:",
            layout=widgets.Layout(width="100%"),
        )
        comp_filter = widgets.Text(
            value="",
            description="Comp filter:",
            placeholder="substring (optional)",
            layout=widgets.Layout(width="100%"),
        )

        def on_comp_filter_change(_ch):
            text = comp_filter.value.strip().lower()
            filtered = (
                comp_opts
                if not text
                else [o for o in comp_opts if text in o[0].lower()]
            )
            comp_dd.options = filtered or comp_opts
            comp_dd.value = (filtered or comp_opts)[0][1]
            refresh_images()

        comp_filter.observe(on_comp_filter_change, names="value")

        ad_names = list_ads(idc)
        ad_a_dd = widgets.Dropdown(
            options=[(n, n) for n in ad_names],
            value=ad_names[0],
            description="AD:",
            layout=widgets.Layout(width="100%"),
        )

        all_shapes = list_shapes(cc)
        shape_filter = widgets.Text(
            value="",
            description="Shape filter:",
            placeholder="e.g. E5.Flex",
            layout=widgets.Layout(width="100%"),
        )

        def filtered_shapes():
            if not shape_filter.value.strip():
                return all_shapes
            s = shape_filter.value.strip().lower()
            return [n for n in all_shapes if s in n.lower()]

        backend_shape_dd = widgets.Dropdown(
            options=[(n, n) for n in filtered_shapes()] or [("No shapes", "")],
            value=(filtered_shapes()[0] if filtered_shapes() else ""),
            description="Backend Shape:",
            layout=widgets.Layout(width="100%"),
        )
        generator_shape_dd = widgets.Dropdown(
            options=[(n, n) for n in filtered_shapes()] or [("No shapes", "")],
            value=(filtered_shapes()[0] if filtered_shapes() else ""),
            description="Generator Shape:",
            layout=widgets.Layout(width="100%"),
        )

        def on_shape_filter(_ch):
            opts = filtered_shapes()
            backend_shape_dd.options = [(n, n) for n in opts] or [("No shapes", "")]
            backend_shape_dd.value = opts[0] if opts else ""
            generator_shape_dd.options = [(n, n) for n in opts] or [("No shapes", "")]
            generator_shape_dd.value = opts[0] if opts else ""
            show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value)
            refresh_images()

        shape_filter.observe(on_shape_filter, names="value")

        show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value)
        backend_shape_dd.observe(
            lambda _ch: (
                show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value),
                refresh_images(),
            ),
            names="value",
        )
        generator_shape_dd.observe(
            lambda _ch: (
                show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value),
                refresh_images(),
            ),
            names="value",
        )

        image_dd = widgets.Dropdown(
            options=[("Select compartment first", "")],
            value="",
            description="Image:",
            layout=widgets.Layout(width="100%"),
        )

        def refresh_images():
            try:
                comp_id = comp_dd.value
                b = backend_shape_dd.value or ""
                g = generator_shape_dd.value or ""
                imgs = list_images(cc, comp_id, b, g)
                items = []
                for im in imgs[:100]:
                    try:
                        created = im.time_created.strftime("%Y-%m-%d")
                    except:
                        created = ""
                    items.append(
                        (
                            f"{im.display_name} — {im.operating_system} {im.operating_system_version} — {created}",
                            im.id,
                        )
                    )
                image_dd.options = items or [
                    ("No compatible Oracle Linux images for current shape(s))", "")
                ]
                image_dd.value = image_dd.options[0][1]
            except Exception:
                image_dd.options = [("Discovery error; try Reload/Reset", "")]
                image_dd.value = ""

        refresh_images()

        _state.update(
            dict(
                comp_dd=comp_dd,
                ad_a_dd=ad_a_dd,
                shape_filter=shape_filter,
                backend_shape_dd=backend_shape_dd,
                generator_shape_dd=generator_shape_dd,
                image_dd=image_dd,
            )
        )

        location_box = section(
            "Location",
            [row(comp_filter), vspace(), row(comp_dd), vspace(), row(ad_a_dd)],
        )

        if _state["backend_ocpus_in"] is None:
            _state["backend_ocpus_in"] = widgets.BoundedIntText(
                value=int(BACKEND_OCPUS),
                min=1,
                max=128,
                step=1,
                description="Backend OCPUs:",
            )
            _state["backend_mem_in"] = widgets.BoundedIntText(
                value=int(BACKEND_MEMORY_GB),
                min=1,
                max=2048,
                step=1,
                description="Backend Memory(GB):",
            )
            _state["generator_ocpus_in"] = widgets.BoundedIntText(
                value=int(GENERATOR_OCPUS),
                min=1,
                max=256,
                step=1,
                description="Generator OCPUs:",
            )
            _state["generator_mem_in"] = widgets.BoundedIntText(
                value=int(GENERATOR_MEMORY_GB),
                min=1,
                max=4096,
                step=1,
                description="Generator Memory(GB):",
            )

        shapes_box = section(
            "Shapes and Flex configuration",
            [
                row(shape_filter),
                vspace(),
                row(backend_shape_dd),
                vspace(),
                row(
                    backend_count_in,
                    _state["backend_ocpus_in"],
                    _state["backend_mem_in"],
                ),
                vspace(),
                row(generator_shape_dd),
                vspace(),
                row(
                    generator_count_in,
                    _state["generator_ocpus_in"],
                    _state["generator_mem_in"],
                ),
            ],
        )

        cps_box = section(
            "CPS settings (used when Test Mode=CPS)",
            [
                row(warm_10k_in, hold_10k_in),
                vspace(),
                row(warm_25k_in, hold_25k_in),
                vspace(),
                row(warm_35k_in, hold_35k_in),
                vspace(),
                row(warm_50k_in, hold_50k_in),
                vspace(),
                row(warm_100k_in, hold_100k_in),
            ],
        )

        tput_box = section(
            "Throughput settings (used when Test Mode=Throughput)",
            [
                row(tput_targets_in),
                vspace(),
                row(tput_warm_in, tput_hold_in),
                vspace(),
                row(tput_payload_size_in),
                vspace(),
                row(tput_payload_bake_in),
                payload_help,
            ],
        )

        cpu_box = section(
            "Generator CPU policy (workers per host)",
            [
                row(workers_per_host_mode_in, fixed_workers_in),
                vspace(),
                row(cpu_reserve_in, min_workers_in, max_workers_in),
                widgets.HTML(
                    "<i>Auto mode: workers/host = clamp(nproc − CPU_RESERVE, MIN, MAX). Memory is uncapped (by design).</i>"
                ),
            ],
        )

        locust_box = section(
            "Locust & UI settings",
            [
                row(test_mode_dd),
                vspace(),
                row(wait_time_in, conn_timeout_in, read_timeout_in, verify_tls_in),
                vspace(),
                row(ui_enable_in, ui_mode_in, ui_port_in, ui_cidr_in),
                vspace(),
                row(health_ep_in),
                vspace(),
                row(throughput_ep_in),
                vspace(),
                row(ssh_cidr_in),
            ],
        )

        image_box = section("Image (compatible with selected shape(s))", row(image_dd))
        dynamic_box.children = [
            location_box,
            shapes_box,
            image_box,
            cps_box,
            tput_box,
            cpu_box,
            locust_box,
        ]
        err_out.clear_output()
    except Exception as e:
        dynamic_box.children = []
        with err_out:
            print("[error] UI rebuild failed:", repr(e))
            print(
                "Use Reset, then try again. If it persists, restart the kernel and run Cells 1–3."
            )


def _parse_size_text_to_bytes(s: str) -> tuple[int, str]:
    s = (s or "").strip().lower()
    if not s:
        raise ValueError("Empty payload size")
    if s.endswith("mb"):
        s = s[:-2] + "m"
    if s.endswith("kb"):
        s = s[:-2] + "k"
    if s.endswith("m"):
        n = float(s[:-1])
        if n <= 0:
            raise ValueError("MB value must be > 0")
        return int(n * 1_000_000), f"{int(n) if n.is_integer() else n}m"
    if s.endswith("k"):
        n = float(s[:-1])
        if n <= 0:
            raise ValueError("KB value must be > 0")
        return int(n * 1_000), f"{int(n) if n.is_integer() else n}k"
    n = float(s)  # bytes
    if n <= 0:
        raise ValueError("Byte value must be > 0")
    return int(n), str(int(n))


def _normalize_bake_list(txt: str) -> list[str]:
    out, seen, norm = [], set(), []
    for tok in (txt or "").split(","):
        tok = tok.strip()
        if not tok:
            continue
        _, label = _parse_size_text_to_bytes(tok)
        out.append(label)
    for l in out:
        if l in seen:
            continue
        seen.add(l)
        norm.append(l)
    return norm


def on_reload_clicked(_):
    rebuild_dynamic_area()


def on_reset_clicked(_):
    region_dd.value = default_region
    rebuild_dynamic_area()


def on_apply_clicked(_):
    try:
        global SELECTED_REGION, REGION, COMPARTMENT_ID, AD_A, IMAGE_ID
        global SSH_PUBLIC_KEY_PATH, SSH_PRIVATE_KEY_PATH, SSH_PUBLIC_KEY_CONTENT
        global BACKEND_COUNT, BACKEND_SHAPE, BACKEND_OCPUS, BACKEND_MEMORY_GB
        global GENERATOR_COUNT, GENERATOR_SHAPE, GENERATOR_OCPUS, GENERATOR_MEMORY_GB
        global TLS_PROTOCOL, HEALTH_ENDPOINT_PATH, THROUGHPUT_ENDPOINT_PATH, SSH_ALLOWED_CIDR
        global LOCUST_WAIT_TIME_SEC, LOCUST_CONNECT_TIMEOUT, LOCUST_READ_TIMEOUT, LOCUST_VERIFY_TLS
        global UI_ENABLE, UI_EXPOSE_MODE, UI_WEB_PORT, UI_ALLOWED_CIDR
        global TEST_MODE, CPS_WARMUPS, CPS_HOLDS
        global TPUT_TARGETS_GBPS_TEXT, TPUT_WARMUP_SEC, TPUT_HOLD_SEC
        global TPUT_PAYLOAD_SIZE_TEXT, TPUT_PAYLOAD_SIZE_BYTES, TPUT_PAYLOAD_SIZE_LABEL
        global TPUT_PAYLOAD_SIZES_TEXT, TPUT_PAYLOAD_BYTES_PER_REQ
        global WORKERS_PER_HOST, CPU_RESERVE, MIN_WORKERS_PER_HOST, MAX_WORKERS_PER_HOST

        SELECTED_REGION = region_dd.value
        REGION = SELECTED_REGION

        comp_dd = _state["comp_dd"]
        ad_a_dd = _state["ad_a_dd"]
        backend_shape_dd = _state["backend_shape_dd"]
        generator_shape_dd = _state["generator_shape_dd"]
        image_dd = _state["image_dd"]

        COMPARTMENT_ID = comp_dd.value
        AD_A = ad_a_dd.value

        BACKEND_COUNT = int(backend_count_in.value)
        GENERATOR_COUNT = int(generator_count_in.value)

        BACKEND_SHAPE = backend_shape_dd.value or ""
        GENERATOR_SHAPE = generator_shape_dd.value or ""
        if BACKEND_SHAPE.endswith(".Flex"):
            BACKEND_OCPUS = float(_state["backend_ocpus_in"].value)
            BACKEND_MEMORY_GB = float(_state["backend_mem_in"].value)
        if GENERATOR_SHAPE.endswith(".Flex"):
            GENERATOR_OCPUS = float(_state["generator_ocpus_in"].value)
            GENERATOR_MEMORY_GB = float(_state["generator_mem_in"].value)

        IMAGE_ID = image_dd.value or ""

        SSH_PUBLIC_KEY_PATH = os.path.expanduser(
            ssh_pub_dd.value or SSH_PUBLIC_KEY_PATH
        )
        SSH_PRIVATE_KEY_PATH = os.path.expanduser(
            ssh_priv_dd.value or SSH_PRIVATE_KEY_PATH
        )
        with open(SSH_PUBLIC_KEY_PATH, "r") as f:
            SSH_PUBLIC_KEY_CONTENT = f.read().strip()

        TLS_PROTOCOL = tls_dd.value
        HEALTH_ENDPOINT_PATH = (health_ep_in.value or "/healthz").strip()
        THROUGHPUT_ENDPOINT_PATH = (throughput_ep_in.value or "/payload_100k").strip()
        SSH_ALLOWED_CIDR = (ssh_cidr_in.value or "0.0.0.0/0").strip()

        LOCUST_WAIT_TIME_SEC = float(wait_time_in.value)
        LOCUST_CONNECT_TIMEOUT = int(conn_timeout_in.value)
        LOCUST_READ_TIMEOUT = int(read_timeout_in.value)
        LOCUST_VERIFY_TLS = bool(verify_tls_in.value)

        UI_ENABLE = bool(ui_enable_in.value)
        UI_EXPOSE_MODE = ui_mode_in.value or "tunnel"
        UI_WEB_PORT = int(ui_port_in.value)
        UI_ALLOWED_CIDR = (ui_cidr_in.value or "0.0.0.0/0").strip()

        # Mode & durations
        TEST_MODE = test_mode_dd.value
        CPS_WARMUPS = {
            10000: int(warm_10k_in.value),
            25000: int(warm_25k_in.value),
            35000: int(warm_35k_in.value),
            50000: int(warm_50k_in.value),
            100000: int(warm_100k_in.value),
        }
        CPS_HOLDS = {
            10000: int(hold_10k_in.value),
            25000: int(hold_25k_in.value),
            35000: int(hold_35k_in.value),
            50000: int(hold_50k_in.value),
            100000: int(hold_100k_in.value),
        }

        TPUT_TARGETS_GBPS_TEXT = (tput_targets_in.value or "1,5,10").strip()
        TPUT_WARMUP_SEC = int(tput_warm_in.value)
        TPUT_HOLD_SEC = int(tput_hold_in.value)

        # Payload sizes
        TPUT_PAYLOAD_SIZE_TEXT = (tput_payload_size_in.value or "100k").strip()
        TPUT_PAYLOAD_SIZES_TEXT = (
            tput_payload_bake_in.value or "4k,5k,10k,50k,100k,256k,1m,5m"
        ).strip()
        TPUT_PAYLOAD_SIZE_BYTES, TPUT_PAYLOAD_SIZE_LABEL = _parse_size_text_to_bytes(
            TPUT_PAYLOAD_SIZE_TEXT
        )
        TPUT_PAYLOAD_BYTES_PER_REQ = int(TPUT_PAYLOAD_SIZE_BYTES)

        # CPU policy
        mode = workers_per_host_mode_in.value
        if mode == "fixed":
            WORKERS_PER_HOST = int(fixed_workers_in.value)
        else:
            WORKERS_PER_HOST = "auto"
        CPU_RESERVE = int(cpu_reserve_in.value)
        MIN_WORKERS_PER_HOST = int(min_workers_in.value)
        MAX_WORKERS_PER_HOST = int(max_workers_in.value)

        # Propagate key items as env for downstream shells
        os.environ["OCI_CONFIG_FILE"] = OCI_CONFIG_FILE
        os.environ["OCI_PROFILE"] = OCI_PROFILE
        os.environ["REGION"] = REGION
        os.environ["COMPARTMENT_ID"] = COMPARTMENT_ID

        # Summary panel
        summary_out.clear_output()
        with summary_out:
            display(
                HTML(
                    f"""
            <div style="border:{BORDER};background:{SECTION_BG};padding:{PAD};">
              <b>Selections applied</b>
              <div style="font-family:ui-monospace; white-space:pre-wrap;">
Region: {REGION}
Compartment: {COMPARTMENT_ID}
AD: {AD_A}
Test Mode: {TEST_MODE}
Backend Shape: {BACKEND_SHAPE} | OCPUs={BACKEND_OCPUS if BACKEND_SHAPE.endswith('.Flex') else '-'} | MemGB={BACKEND_MEMORY_GB if BACKEND_SHAPE.endswith('.Flex') else '-'} | Count={BACKEND_COUNT}
Generator Shape: {GENERATOR_SHAPE} | OCPUs={GENERATOR_OCPUS if GENERATOR_SHAPE.endswith('.Flex') else '-'} | MemGB={GENERATOR_MEMORY_GB if GENERATOR_SHAPE.endswith('.Flex') else '-'} | Count={GENERATOR_COUNT}
Image: {IMAGE_ID or '(unset)'}
TLS protocol: {TLS_PROTOCOL}
Health path: {HEALTH_ENDPOINT_PATH}
Throughput base path: {THROUGHPUT_ENDPOINT_PATH}
Throughput payload (run): {TPUT_PAYLOAD_SIZE_LABEL} (~{TPUT_PAYLOAD_SIZE_BYTES} bytes)
Throughput payloads to bake at build: {_normalize_bake_list(TPUT_PAYLOAD_SIZES_TEXT)}
Locust: wait(s)={LOCUST_WAIT_TIME_SEC} | connect(ms)={LOCUST_CONNECT_TIMEOUT} | read(ms)={LOCUST_READ_TIMEOUT} | verify_tls={LOCUST_VERIFY_TLS}
UI: enable={UI_ENABLE} | mode={UI_EXPOSE_MODE} | port={UI_WEB_PORT} | allowed_cidr={UI_ALLOWED_CIDR}
CPS Warmups: {CPS_WARMUPS}
CPS Holds:   {CPS_HOLDS}
TPUT targets (Gbps): {TPUT_TARGETS_GBPS_TEXT}   (RPS ≈ Gbps*1e9/8/{TPUT_PAYLOAD_SIZE_BYTES})
TPUT warmup/hold (s): {TPUT_WARMUP_SEC} / {TPUT_HOLD_SEC}
Workers/host mode: {WORKERS_PER_HOST} | CPU_RESERVE={CPU_RESERVE} | MIN={MIN_WORKERS_PER_HOST} | MAX={MAX_WORKERS_PER_HOST}
API key path: {OCI_PRIVATE_KEY_PATH}
SSH pub: {SSH_PUBLIC_KEY_PATH}
SSH priv: {SSH_PRIVATE_KEY_PATH}
              </div>
            </div>
            """
                )
            )
        print("Cell 3 complete: Selections applied.")
        print(
            "- To PROVISION infra: run Cell 4 → 7 (backends will bake payload files)."
        )
        print("- If infra already exists: run Cells 8 → 10 for outputs & sanity.")
        print(
            "- Then headless: Cell 11 (orchestration), Cell 16 (warm-up), Cell 17 (suite), Cell 18 (metrics)."
        )
        print(
            "- UI path: Cell 12 (starts UI master/workers; honors Test Mode & CPU policy)."
        )
    except Exception as e:
        err_out.clear_output()
        with err_out:
            print("[error] Apply failed:", repr(e))


reload_btn.on_click(on_reload_clicked)
reset_btn.on_click(on_reset_clicked)
apply_btn.on_click(on_apply_clicked)
region_dd.observe(rebuild_dynamic_area, names="value")

rebuild_dynamic_area()
container = widgets.VBox(
    [
        section("Location", [row(region_dd, reload_btn, reset_btn)]),
        dynamic_box,
        section("Keys (API + SSH)", [row(ssh_pub_dd), vspace(), row(ssh_priv_dd)]),
        section(
            "Locust/Test params",
            [
                row(tls_dd),
                vspace(),
                row(health_ep_in),
                vspace(),
                row(throughput_ep_in),
                vspace(),
                row(ssh_cidr_in),
            ],
        ),
        widgets.HBox([apply_btn], layout=widgets.Layout(justify_content="center")),
        err_out,
        summary_out,
    ],
    layout=widgets.Layout(width="100%", max_width=CONTAINER_MAX_W, margin="0 auto"),
)
display(container)

print("Cell 3 loaded: Make selections and click 'Apply Selections'.")
print("NEXT: Build infra (Cells 4–7) or go to outputs (Cells 8–10), then run Cell 11.")

In [ ]:
# Cell 4 — Objective: Write cloud-init scripts (Backends TLS NGINX; Generators Locust)

import os

os.makedirs("cloud-init", exist_ok=True)


def _dd_lines_for_payloads(sizes_csv: str) -> str:
    lines = []

    def _one(tok: str):
        tok = tok.strip().lower()
        if not tok:
            return
        if tok.endswith("mb"):
            tok2 = tok[:-2] + "m"
        elif tok.endswith("kb"):
            tok2 = tok[:-2] + "k"
        else:
            tok2 = tok
        if tok2.endswith("m"):
            n = tok2[:-1]
            lines.append(
                f"dd if=/dev/zero of=/usr/share/nginx/html/payload_{n}m bs=1MB count={n} status=none || true"
            )
        elif tok2.endswith("k"):
            n = tok2[:-1]
            lines.append(
                f"dd if=/dev/zero of=/usr/share/nginx/html/payload_{n}k bs=1KB count={n} status=none || true"
            )
        else:
            lines.append(
                f"head -c {tok2} /dev/zero > /usr/share/nginx/html/payload_{tok2}b || true"
            )

    for t in (TPUT_PAYLOAD_SIZES_TEXT or "").split(","):
        _one(t)
    if not lines:
        lines.append(
            "dd if=/dev/zero of=/usr/share/nginx/html/payload_100k bs=1KB count=100 status=none || true"
        )
    return "\n".join(lines)


_dd_payload_block = _dd_lines_for_payloads(TPUT_PAYLOAD_SIZES_TEXT)

backend_cloud_init_template = """#!/bin/bash
if command -v firewall-cmd >/dev/null 2>&1; then
  systemctl stop firewalld || true
  systemctl disable firewalld || true
fi
if systemctl list-unit-files | grep -q oracle-cloud-agent.service; then
  systemctl enable --now oracle-cloud-agent || true
fi
cat <<EOF >/etc/sysctl.d/99-net-tuning.conf
net.core.somaxconn=262144
net.core.netdev_max_backlog=500000
net.ipv4.tcp_max_syn_backlog=262144
net.ipv4.ip_local_port_range=1024 65535
net.ipv4.tcp_fin_timeout=15
net.ipv4.tcp_tw_reuse=1
net.core.rmem_max=33554432
net.core.wmem_max=33554432
net.ipv4.tcp_syncookies=1
net.ipv4.tcp_fastopen=3
EOF
sysctl --system || true
echo "* - nofile 1048576" | tee -a /etc/security/limits.conf

dnf -y makecache || true
dnf -y install nginx openssl || true

cat >/etc/nginx/nginx.conf <<'NGINX'
user  nginx;
worker_processes auto;
worker_rlimit_nofile 1048576;
events { worker_connections 131072; multi_accept on; accept_mutex off; }
http {
    include /etc/nginx/mime.types;
    default_type application/octet-stream;
    sendfile on; tcp_nopush on; tcp_nodelay on;
    keepalive_timeout 15;
    client_body_timeout 10; client_header_timeout 10; send_timeout 10;
    ssl_session_cache shared:SSL:512m; ssl_session_timeout 10m; ssl_session_tickets on;
    server_tokens off; access_log off;
    include /etc/nginx/conf.d/*.conf;
}
NGINX

# Pre-create payload files for throughput (e.g., /payload_4k, /payload_1m)
__DD_PAYLOAD_BLOCK__

install -d -m 0755 /etc/nginx/certs
openssl ecparam -genkey -name prime256v1 -out /etc/nginx/certs/backend.key
openssl req -x509 -new -key /etc/nginx/certs/backend.key -subj "/CN=backend.local" -days 3650 -out /etc/nginx/certs/backend.crt
chmod 600 /etc/nginx/certs/backend.key
chmod 644 /etc/nginx/certs/backend.crt

mkdir -p /etc/systemd/system/nginx.service.d
cat >/etc/systemd/system/nginx.service.d/override.conf <<'EOF'
[Service]
LimitNOFILE=1048576
LimitNPROC=65536
EOF
systemctl daemon-reload || true

cat >/etc/nginx/conf.d/direct.conf <<'EOF'
server {
    listen 443 ssl reuseport backlog=131072 fastopen=256;
    server_name _;
    access_log off;

    ssl_certificate     /etc/nginx/certs/backend.crt;
    ssl_certificate_key /etc/nginx/certs/backend.key;
    ssl_protocols       TLSv1.3 TLSv1.2;
    ssl_ecdh_curve      prime256v1;
    ssl_buffer_size     4k;

    location = /healthz { return 200 "ok\n"; }
    # Throughput payloads (static files created above), e.g., /payload_4k, /payload_100k, /payload_1m
    location /payload_ { root /usr/share/nginx/html; }
    location /          { return 200 "ok\n"; }
}
EOF

rm -f /etc/nginx/conf.d/default.conf
nginx -t && systemctl enable --now nginx
"""

backend_cloud_init = backend_cloud_init_template.replace(
    "__DD_PAYLOAD_BLOCK__", _dd_payload_block
)

generator_cloud_init = r"""#!/bin/bash
if command -v firewall-cmd >/dev/null 2>&1; then
  systemctl stop firewalld || true
  systemctl disable firewalld || true
fi
cat <<EOF >/etc/sysctl.d/99-freewheel.conf
net.core.somaxconn=65535
net.core.netdev_max_backlog=250000
net.ipv4.tcp_max_syn_backlog=262144
net.ipv4.ip_local_port_range=1024 65535
net.ipv4.tcp_fin_timeout=15
net.ipv4.tcp_tw_reuse=1
fs.file-max=1000000
EOF
sysctl --system || true
echo "* - nofile 1048576" >> /etc/security/limits.conf

dnf -y makecache || true
dnf -y install python3 python3-pip tmux curl || true
pip3 install --no-cache-dir --upgrade pip || true
pip3 install --no-cache-dir locust || true

mkdir -p /home/opc/locustwork/results
chown -R opc:opc /home/opc/locustwork

cat > /home/opc/locustwork/locustfile.py <<'PY'
import os
from locust import HttpUser, task, constant

VERIFY_TLS        = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = float(os.environ.get("LOCUST_CONNECT_TIMEOUT_S", "8"))
READ_TIMEOUT_S    = float(os.environ.get("LOCUST_READ_TIMEOUT_S", "15"))
WAIT_TIME_S       = float(os.environ.get("LOCUST_WAIT_TIME_S", "1.0"))
MODE              = os.environ.get("LOCUST_MODE", "cps").lower()  # "cps" or "throughput"
HEALTH_PATH       = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
THROUGHPUT_PATH   = os.environ.get("LOCUST_THROUGHPUT_PATH", "/payload_100k")

class CpsUser(HttpUser):
    wait_time = constant(WAIT_TIME_S)

    @task
    def do_request(self):
        path    = HEALTH_PATH if MODE == "cps" else THROUGHPUT_PATH
        headers = {"Connection": "close"} if MODE == "cps" else {}  # keep-alive by default for throughput
        self.client.get(
            path,
            headers=headers,
            verify=VERIFY_TLS,
            timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
            name=("cps_req" if MODE == "cps" else "throughput_req"),
        )
PY

chown -R opc:opc /home/opc/locustwork
echo READY
"""

with open("cloud-init/backend.sh.tftpl", "w") as f:
    f.write(backend_cloud_init)
with open("cloud-init/generator.sh", "w") as f:
    f.write(generator_cloud_init)

print(
    "Cell 4 complete: Wrote cloud-init/backend.sh.tftpl and UPDATED cloud-init/generator.sh (direct TLS; payloads baked)."
)
print(
    "NEXT: Run Cell 5 to generate Terraform, Cell 6 to write tfvars, then Cell 7 to apply (if provisioning)."
)

In [ ]:
# Cell 5 — Objective: Terraform using NSGs, VCN/subnets; NO Load Balancer (direct generator->backend HTTPS)

backend_shape_config_block = ""
if (BACKEND_SHAPE or "").endswith(".Flex"):
    backend_shape_config_block = """
  shape_config {
    ocpus         = %s
    memory_in_gbs = %s
  }""" % (
        BACKEND_OCPUS,
        BACKEND_MEMORY_GB,
    )

generator_shape_config_block = ""
if (GENERATOR_SHAPE or "").endswith(".Flex"):
    generator_shape_config_block = """
  shape_config {
    ocpus         = %s
    memory_in_gbs = %s
  }""" % (
        GENERATOR_OCPUS,
        GENERATOR_MEMORY_GB,
    )

terraform_config_template = """terraform {
  required_providers {
    oci = {
      source  = "oracle/oci"
      version = ">= 5.39.0"
    }
  }
}

provider "oci" {
  config_file_profile = var.oci_profile
  region              = var.region
}

# Variables
variable "oci_profile" {}
variable "region" {}
variable "compartment_id" {}
variable "ad_a" {}
variable "image_id" {}
variable "ssh_public_key_content" {}
variable "ssh_private_key_path" {}

variable "backend_count" {}
variable "backend_shape" {}
variable "generator_count" {}
variable "generator_shape" {}

variable "ssh_allowed_cidr" {}

variable "bastion_plugin_name" {
  type    = string
  default = "Bastion"
}

# VCN
resource "oci_core_vcn" "vcn" {
  cidr_block     = "10.0.0.0/16"
  compartment_id = var.compartment_id
  display_name   = "direct-test-vcn"
}

# Gateways
resource "oci_core_internet_gateway" "igw" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "direct-test-igw"
}
resource "oci_core_nat_gateway" "nat" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "direct-test-nat"
}

# Route Tables
resource "oci_core_route_table" "rt_public" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "direct-test-rt-public"
  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }
}
resource "oci_core_route_table" "rt_private" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "direct-test-rt-private"
  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_nat_gateway.nat.id
  }
}

# NSGs
resource "oci_core_network_security_group" "nsg_backends" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-backends"
}
resource "oci_core_network_security_group" "nsg_generators" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-generators"
}

# NSG Rules
# Backends: allow HTTPS (443) and SSH (22) from generators NSG only (SSH via jump host)
resource "oci_core_network_security_group_security_rule" "backends_ingress_443_from_gens" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_generators.id
  tcp_options {
    destination_port_range {
      min = 443
      max = 443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "backends_ingress_22_from_gens" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_generators.id
  tcp_options {
    destination_port_range {
      min = 22
      max = 22
    }
  }
}

resource "oci_core_network_security_group_security_rule" "backends_egress_all" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
}

# Generators: SSH ingress from CIDR, intra-NSG for locust (5557-5558), egress all
resource "oci_core_network_security_group_security_rule" "gens_ingress_ssh" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = var.ssh_allowed_cidr
  tcp_options {
    destination_port_range {
      min = 22
      max = 22
    }
  }
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_locust_master" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_generators.id
  tcp_options {
    destination_port_range {
      min = 5557
      max = 5558
    }
  }
}

resource "oci_core_network_security_group_security_rule" "gens_egress_all" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
}

# Subnets
resource "oci_core_subnet" "backends_priv" {
  cidr_block                 = "10.0.2.0/24"
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "direct-test-backends-priv"
  route_table_id             = oci_core_route_table.rt_private.id
  prohibit_public_ip_on_vnic = true
}
resource "oci_core_subnet" "gens_pub" {
  cidr_block                 = "10.0.3.0/24"
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "direct-test-generators-pub"
  route_table_id             = oci_core_route_table.rt_public.id
  prohibit_public_ip_on_vnic = false
}

# Compute: backends
resource "oci_core_instance" "backend" {
  count               = var.backend_count
  availability_domain = var.ad_a
  compartment_id      = var.compartment_id
  shape               = var.backend_shape__BACKEND_SHAPE_CONFIG__
  source_details {
    source_type = "image"
    source_id   = var.image_id
  }
  create_vnic_details {
    subnet_id        = oci_core_subnet.backends_priv.id
    assign_public_ip = false
    nsg_ids          = [oci_core_network_security_group.nsg_backends.id]
  }
  agent_config {
    are_all_plugins_disabled = false
    is_management_disabled   = false
    is_monitoring_disabled   = false
    plugins_config {
      name          = var.bastion_plugin_name
      desired_state = "ENABLED"
    }
  }
  display_name = "direct-backend-${count.index}"
  metadata = {
    ssh_authorized_keys = var.ssh_public_key_content
    user_data           = base64encode(file("${path.module}/cloud-init/backend.sh.tftpl"))
  }
}

# Compute: generators
resource "oci_core_instance" "generator" {
  count               = var.generator_count
  availability_domain = var.ad_a
  compartment_id      = var.compartment_id
  shape               = var.generator_shape__GENERATOR_SHAPE_CONFIG__
  source_details {
    source_type = "image"
    source_id   = var.image_id
  }
  create_vnic_details {
    subnet_id        = oci_core_subnet.gens_pub.id
    assign_public_ip = true
    nsg_ids          = [oci_core_network_security_group.nsg_generators.id]
  }
  display_name = "direct-gen-${count.index}"
  metadata = {
    ssh_authorized_keys = var.ssh_public_key_content
    user_data           = filebase64("${path.module}/cloud-init/generator.sh")
  }
}

# Outputs
output "gen_public_ips"      { value = [for i in oci_core_instance.generator : i.public_ip] }
output "backend_private_ips" { value = [for i in oci_core_instance.backend   : i.private_ip] }
"""

terraform_config = terraform_config_template.replace(
    "__BACKEND_SHAPE_CONFIG__", backend_shape_config_block
).replace("__GENERATOR_SHAPE_CONFIG__", generator_shape_config_block)

with open("main.tf", "w") as f:
    f.write(terraform_config)

print(
    "Cell 5 complete: Terraform configuration saved to main.tf (no LB; corrected tcp_options blocks)."
)
print("NEXT: Run Cell 6 to write terraform.tfvars using selections from Cell 3.")

In [ ]:
# Cell 6 — Objective: Write terraform.tfvars using dashboard selections and paths only (no secrets).

import os

if not IMAGE_ID:
    raise ValueError("IMAGE_ID is not set. Run Cell 3 and select an Image.")
if not BACKEND_SHAPE:
    raise ValueError("BACKEND_SHAPE is not set. Run Cell 3 and select a Backend Shape.")
if not GENERATOR_SHAPE:
    raise ValueError(
        "GENERATOR_SHAPE is not set. Run Cell 3 and select a Generator Shape."
    )
if not SSH_PUBLIC_KEY_PATH or not os.path.exists(SSH_PUBLIC_KEY_PATH):
    raise ValueError(
        f"SSH public key file missing. Pick a valid key in Cell 3. Current: {SSH_PUBLIC_KEY_PATH}"
    )
if not SSH_PRIVATE_KEY_PATH or not os.path.exists(SSH_PRIVATE_KEY_PATH):
    raise ValueError(
        f"SSH private key file missing. Pick a valid key in Cell 3. Current: {SSH_PRIVATE_KEY_PATH}"
    )

with open(os.path.expanduser(SSH_PUBLIC_KEY_PATH), "r") as f:
    ssh_public_key_content = f.read().strip()

tfvars = f"""
oci_profile            = "{OCI_PROFILE}"
region                 = "{REGION}"
compartment_id         = "{COMPARTMENT_ID}"
ad_a                   = "{AD_A}"
image_id               = "{IMAGE_ID}"
ssh_public_key_content = "{ssh_public_key_content}"
ssh_private_key_path   = "{os.path.expanduser(SSH_PRIVATE_KEY_PATH)}"

backend_count   = {int(BACKEND_COUNT)}
backend_shape   = "{BACKEND_SHAPE}"

generator_count = {int(GENERATOR_COUNT)}
generator_shape = "{GENERATOR_SHAPE}"

ssh_allowed_cidr = "{SSH_ALLOWED_CIDR}"

bastion_plugin_name = "Bastion"
""".lstrip()

with open("terraform.tfvars", "w") as f:
    f.write(tfvars)

print("Cell 6 complete: terraform.tfvars written.")
print("NEXT: Run Cell 7 to terraform init/apply and build the environment.")

In [ ]:
# Cell 7 — Objective: Initialize and apply Terraform (fresh build)

!terraform init
!terraform apply -auto-approve -var-file=terraform.tfvars

print("Cell 7 complete: Terraform apply finished.")
print("NEXT: Run Cell 8 to capture outputs (Generators/Backends). Then run Cell 10 for sanity.")


In [ ]:
# Cell 8 — Objective: Extract instance outputs and select backend target.

import json, subprocess


def get_tf_outputs():
    r = subprocess.run(["terraform", "output", "-json"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr)
    o = json.loads(r.stdout or "{}")
    val = lambda k: o.get(k, {}).get("value")
    return (val("gen_public_ips"), val("backend_private_ips"))


GEN_IPS, BACKEND_IPS = get_tf_outputs()
BACKEND_IPS = BACKEND_IPS or []
BACKEND_TARGET_INDEX = max(0, min(BACKEND_TARGET_INDEX, max(0, len(BACKEND_IPS) - 1)))
TARGET_BACKEND = BACKEND_IPS[BACKEND_TARGET_INDEX] if BACKEND_IPS else ""

print("Generators (public):", GEN_IPS)
print("Backends (private):", BACKEND_IPS)
print("Selected backend target (by index):", BACKEND_TARGET_INDEX, "=>", TARGET_BACKEND)

print("Cell 8 complete: Outputs captured (target backend selected).")
print("NEXT: Run Cell 9 to load SSH helpers, then Cell 10 for sanity HTTP.")

In [ ]:
# Cell 9 — Objective: SSH helpers and system snapshot utilities (Paramiko; jump via generator for backends)

import os, time, json
import paramiko
from datetime import datetime, timezone

SSH_KEY_PASSPHRASE = os.environ.get("SSH_KEY_PASSPHRASE", None)
SNAPSHOT_TIMEOUT_SEC = int(
    os.environ.get("SNAPSHOT_TIMEOUT_SEC", "90")
)  # generous under load


def _load_pkey(key_path: str, passphrase: str | None):
    kpath = os.path.expanduser(key_path)
    if not os.path.exists(kpath):
        raise FileNotFoundError(f"SSH private key not found: {kpath}")
    excs = []
    for KeyClass in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return KeyClass.from_private_key_file(kpath, password=passphrase)
        except Exception as e:
            excs.append(repr(e))
    raise paramiko.SSHException(
        "Could not load SSH private key. Path: %s\n - %s" % (kpath, "\n - ".join(excs))
    )


def ssh_exec(
    host, user="opc", key_path=SSH_PRIVATE_KEY_PATH, command="echo ok", timeout=600
):
    pkey = _load_pkey(key_path, SSH_KEY_PASSPHRASE)
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username=user, pkey=pkey, timeout=30)
    stdin, stdout, stderr = ssh.exec_command(command, timeout=timeout)
    try:
        out = stdout.read().decode("utf-8", errors="ignore")
    except Exception:
        out = ""
    try:
        err = stderr.read().decode("utf-8", errors="ignore")
    except Exception:
        err = ""
    try:
        rc = stdout.channel.recv_exit_status()
    except Exception:
        rc = None
    ssh.close()
    return out.strip(), err.strip(), rc


# Paramiko ProxyJump-style helper: run command on target_private_ip via public jump_host
def ssh_exec_via_jump(
    jump_host_public,
    target_private_ip,
    user="opc",
    key_path=SSH_PRIVATE_KEY_PATH,
    command="echo ok",
    timeout=600,
):
    pkey = _load_pkey(key_path, SSH_KEY_PASSPHRASE)

    # Connect to jump (generator public IP)
    jump = paramiko.SSHClient()
    jump.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    jump.connect(hostname=jump_host_public, username=user, pkey=pkey, timeout=30)

    try:
        # Open direct-tcpip channel from jump to backend:22
        jump_transport = jump.get_transport()
        dest_addr = (target_private_ip, 22)
        local_addr = ("127.0.0.1", 0)
        channel = jump_transport.open_channel("direct-tcpip", dest_addr, local_addr)

        # Connect SSHClient to backend over that channel
        backend = paramiko.SSHClient()
        backend.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        backend.connect(
            hostname=target_private_ip,
            username=user,
            pkey=pkey,
            sock=channel,
            timeout=30,
        )

        try:
            stdin, stdout, stderr = backend.exec_command(command, timeout=timeout)
            try:
                out = stdout.read().decode("utf-8", errors="ignore")
            except Exception:
                out = ""
            try:
                err = stderr.read().decode("utf-8", errors="ignore")
            except Exception:
                err = ""
            try:
                rc = stdout.channel.recv_exit_status()
            except Exception:
                rc = None
            return out.strip(), err.strip(), rc
        finally:
            backend.close()
    finally:
        try:
            jump.close()
        except Exception:
            pass


# Non-blocking launcher for nohup/tmux commands (prevents Paramiko read timeouts)
def ssh_exec_quick(host, command):
    pkey = _load_pkey(SSH_PRIVATE_KEY_PATH, SSH_KEY_PASSPHRASE)
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
    transport = ssh.get_transport()
    chan = transport.open_session()
    chan.exec_command(command)
    time.sleep(0.5)
    try:
        chan.close()
    except Exception:
        pass
    ssh.close()


def ssh_exec_many(hosts, cmd, timeout=1200):
    return {h: ssh_exec(h, command=cmd, timeout=timeout) for h in (hosts or [])}


def system_snapshot(host):
    cmds = {
        "time_utc": "date -u +%FT%TZ",
        "loadavg": "cat /proc/loadavg || true",
        "sockstat": "cat /proc/net/sockstat || true",
        "ss_summary": "ss -s || true",
        "established_443": "ss -Htan state established '( sport = :443 or dport = :443 )' | wc -l || true",
        "nproc": "nproc || getconf _NPROCESSORS_ONLN || echo 1",
        "uname": "uname -a || true",
    }
    out = {}
    for k, cmd in cmds.items():
        o, e, rc = ssh_exec(
            host, command=f"bash -lc '{cmd}'", timeout=SNAPSHOT_TIMEOUT_SEC
        )
        out[k] = o.strip()
    return out


def system_snapshot_backend(backend_priv_ip, jump_public_ip):
    cmds = {
        "time_utc": "date -u +%FT%TZ",
        "loadavg": "cat /proc/loadavg || true",
        "sockstat": "cat /proc/net/sockstat || true",
        "ss_summary": "ss -s || true",
        "established_443": "ss -Htan state established '( sport = :443 or dport = :443 )' | wc -l || true",
        "nproc": "nproc || getconf _NPROCESSORS_ONLN || echo 1",
        "uname": "uname -a || true",
    }
    out = {}
    for k, cmd in cmds.items():
        o, e, rc = ssh_exec_via_jump(
            jump_public_ip,
            backend_priv_ip,
            command=f"bash -lc '{cmd}'",
            timeout=SNAPSHOT_TIMEOUT_SEC,
        )
        out[k] = (o or "").strip()
    return out


def collect_snapshots(gen_hosts, backend_hosts, label_ts, phase, base_dir=OUTPUT_DIR):
    import json, os

    # Pick a jump host for backends: prefer MASTER, else first generator
    jump_host = None
    try:
        if "MASTER" in globals() and MASTER:
            jump_host = MASTER
        elif gen_hosts:
            jump_host = gen_hosts[0]
    except Exception:
        jump_host = gen_hosts[0] if gen_hosts else None

    d = os.path.join(base_dir, f"snapshots_{label_ts}_{phase}")
    os.makedirs(d, exist_ok=True)

    # Generators (public SSH from local)
    for h in gen_hosts or []:
        try:
            snap = system_snapshot(h)
            with open(os.path.join(d, f"generator_{h}.json"), "w") as f:
                json.dump(snap, f, indent=2)
        except Exception as e:
            print(f"Snapshot({phase}) generator {h} error:", e)

    # Backends (private IPs) — snapshot via jump host (generator)
    for h in backend_hosts or []:
        try:
            if not jump_host:
                print(f"Snapshot({phase}) backend {h} skipped: no available jump host.")
                continue
            snap = system_snapshot_backend(h, jump_host)
            with open(os.path.join(d, f"backend_{h}.json"), "w") as f:
                json.dump(snap, f, indent=2)
        except Exception as e:
            print(f"Snapshot({phase}) backend {h} error:", e)

    print(f"Saved system snapshots to: {d}")


print(
    "Cell 9 complete: SSH + snapshot helpers loaded (backend snapshots via generator jump)."
)
print("NEXT: Run Cell 10 for generator→backend sanity checks.")

In [ ]:
# Cell 10 — Objective: Sanity checks (+ basic connectivity)


def curl_status_from_gen(host, url, timeout=3):
    cmd = f"curl -skI --connect-timeout {timeout} --max-time {timeout} {url} | head -n1 || true"
    out, err, rc = ssh_exec(host, command=cmd)
    line = (out or "").strip()
    return ((line if line else "blocked/timeout"), (err or ""), rc)


if not TARGET_BACKEND:
    print("No backend target available (run Cell 8).")
else:
    print("Sanity from each generator to Backend target (HTTPS):")
    for h in GEN_IPS or []:
        url = f"https://{TARGET_BACKEND}{HEALTH_ENDPOINT_PATH}"
        st_out, st_err, _ = curl_status_from_gen(h, url, timeout=3)
        print(f"{h} => {url} => status: {st_out} | err: {st_err or '(no stderr)'}")

print("Cell 10 complete: Sanity check executed.")
print("NEXT: Run Cell 11 to load orchestration (auto workers per CPU).")

In [ ]:
# Cell 11 — Objective: Distributed Locust orchestration (supports CPS & Throughput, single-backend target)
# Adds pre-kill of existing workers per host to prevent duplicate workers.

import os, time, io, paramiko, glob
import pandas as pd
from datetime import datetime, timezone, timedelta

MASTER = GEN_IPS[0] if GEN_IPS else None
WORKERS = GEN_IPS[1:] if GEN_IPS and len(GEN_IPS) > 1 else []
WORKDIR_REMOTE = LOCUST_WORKDIR
RESULTS_DIR_REMOTE = f"{WORKDIR_REMOTE}/results"

TMUX_UI_MASTER = "locust_ui_master"
TMUX_HEADLESS_MASTER = "locust_headless"


def _export_env(mode="cps"):
    mode = (mode or "cps").lower()
    assert mode in ("cps", "throughput")
    return (
        f"export LOCUST_MODE={mode}; "
        f"export LOCUST_HEALTH_PATH='{HEALTH_ENDPOINT_PATH}'; "
        f"export LOCUST_THROUGHPUT_PATH='{THROUGHPUT_ENDPOINT_PATH}'; "
        f"export LOCUST_VERIFY_TLS={'true' if LOCUST_VERIFY_TLS else 'false'}; "
        f"export LOCUST_CONNECT_TIMEOUT_S={LOCUST_CONNECT_TIMEOUT/1000.0}; "
        f"export LOCUST_READ_TIMEOUT_S={LOCUST_READ_TIMEOUT/1000.0}; "
        f"export LOCUST_WAIT_TIME_S={LOCUST_WAIT_TIME_SEC}; "
        f"export PATH=$HOME/.local/bin:/usr/local/bin:/usr/bin:/bin:$PATH; "
    )


def _stop_tmux_session(host, session_name):
    cmd = rf"bash -lc 'tmux has-session -t {session_name} 2>/dev/null && tmux kill-session -t {session_name} || true'"
    ssh_exec(host, command=cmd, timeout=20)


def _ensure_workspace_and_locust(host):
    setup = rf"""bash -lc '
set -e
mkdir -p {RESULTS_DIR_REMOTE}
python3 -m pip install --no-cache-dir --upgrade --user pip || true
python3 -m pip show locust >/dev/null 2>&1 || python3 -m pip install --no-cache-dir --user locust
python3 - <<PY
try:
    import locust
    print("LOCUST_OK", locust.__version__)
except Exception as e:
    print("LOCUST_FAIL", e)
PY
echo READY
'"""
    out, err, rc = ssh_exec(host, command=setup, timeout=240)
    if "LOCUST_OK" not in (out or ""):
        raise RuntimeError(
            f"{host}: locust not available for opc; setup output:\n{out}\n{err}"
        )

    LOCUSTFILE_CODE = r"""import os
from locust import HttpUser, task, constant
VERIFY_TLS        = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = float(os.environ.get("LOCUST_CONNECT_TIMEOUT_S", "8"))
READ_TIMEOUT_S    = float(os.environ.get("LOCUST_READ_TIMEOUT_S", "15"))
WAIT_TIME_S       = float(os.environ.get("LOCUST_WAIT_TIME_S", "1.0"))
MODE              = os.environ.get("LOCUST_MODE", "cps").lower()
HEALTH_PATH       = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
THROUGHPUT_PATH   = os.environ.get("LOCUST_THROUGHPUT_PATH", "/payload_100k")
class CpsUser(HttpUser):
    wait_time = constant(WAIT_TIME_S)
    @task
    def do_request(self):
        path    = HEALTH_PATH if MODE == "cps" else THROUGHPUT_PATH
        headers = {"Connection": "close"} if MODE == "cps" else {}
        self.client.get(path, headers=headers, verify=VERIFY_TLS,
                        timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                        name=("cps_req" if MODE == "cps" else "throughput_req"))"""
    pkey = _load_pkey(SSH_PRIVATE_KEY_PATH, os.environ.get("SSH_KEY_PASSPHRASE", None))
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
    try:
        transport = ssh.get_transport()
        sftp = paramiko.SFTPClient.from_transport(transport)
        try:
            sftp.stat(WORKDIR_REMOTE)
        except FileNotFoundError:
            sftp.mkdir(WORKDIR_REMOTE)
        with sftp.file(f"{WORKDIR_REMOTE}/locustfile.py", "w") as f:
            f.set_pipelined(True)
            f.write(LOCUSTFILE_CODE)
        sftp.chmod(f"{WORKDIR_REMOTE}/locustfile.py", 0o644)
        sftp.close()
    finally:
        ssh.close()


def _tmux_session_exists(host, session_name):
    out, err, rc = ssh_exec(
        host,
        command=rf"bash -lc 'tmux has-session -t {session_name} 2>/dev/null && echo YES || echo NO'",
    )
    return (out or "").strip() == "YES"


def _get_private_ip(host):
    if MASTER_PRIVATE_IP_OVERRIDE:
        return MASTER_PRIVATE_IP_OVERRIDE
    cmd = r"""bash -lc '
get_ip() {
  if curl -fsS -H "Authorization: Bearer Oracle" http://169.254.169.254/opc/v2/vnics/ >/tmp/vn.json 2>/dev/null; then :; 
  elif curl -fsS http://169.254.169.254/opc/v1/vnics/ >/tmp/vn.json 2>/dev/null; then :; 
  else : > /tmp/vn.json; fi
  IP=$(tr -d "\n" </tmp/vn.json | sed -n '\''s/.*"privateIp"[[:space:]]*:[[:space:]]*"\([0-9.]*\)".*/\1/p'\'' | head -1)
  [ -n "$IP" ] && echo "$IP" && return 0
  IP=$(ip -4 route get 1.1.1.1 2>/dev/null | awk '\''{for(i=1;i<=NF;i++) if($i=="src") print $(i+1)}'\'' | head -1)
  [ -n "$IP" ] && echo "$IP" && return 0
  IP=$(hostname -I 2>/dev/null | awk '\''{print $1}'\'' | head -1)
  [ -n "$IP" ] && echo "$IP" && return 0
  IP=$(ip -4 addr show scope global 2>/dev/null | awk '\''/inet /{print $2}'\'' | cut -d/ -f1 | head -1)
  [ -n "$IP" ] && echo "$IP" || echo ""
}
get_ip
'"""
    out, err, rc = ssh_exec(host, command=cmd, timeout=20)
    return (out or "").strip()


def _detect_nproc(host):
    cmd = r"bash -lc 'nproc 2>/dev/null || getconf _NPROCESSORS_ONLN 2>/dev/null || echo 1'"
    out, err, rc = ssh_exec(host, command=cmd, timeout=10)
    try:
        n = int((out or "1").strip())
        return max(1, n)
    except Exception:
        return 1


def _count_remote_workers(host):
    cmd = r"""bash -lc "ps -eo cmd | grep -E 'python3 -m locust .*--worker( |$)|locust .*--worker( |$)' | grep -v grep | wc -l" """
    out, err, rc = ssh_exec(host, command=cmd, timeout=10)
    try:
        return int((out or "0").strip())
    except Exception:
        return 0


def _upload_text_file(host, remote_path, content, mode=0o755):
    pkey = _load_pkey(SSH_PRIVATE_KEY_PATH, os.environ.get("SSH_KEY_PASSPHRASE", None))
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
    try:
        sftp = ssh.open_sftp()
        try:
            sftp.stat(os.path.dirname(remote_path))
        except FileNotFoundError:
            try:
                sftp.mkdir(os.path.dirname(remote_path))
            except Exception:
                pass
        with sftp.file(remote_path, "w") as f:
            f.write(content)
        sftp.chmod(remote_path, mode)
        sftp.close()
    finally:
        ssh.close()


def _write_and_run_bootstrap(host, script_text):
    pkey = _load_pkey(SSH_PRIVATE_KEY_PATH, os.environ.get("SSH_KEY_PASSPHRASE", None))
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
    try:
        sftp = ssh.open_sftp()
        path = f"{WORKDIR_REMOTE}/start_workers.sh"
        try:
            sftp.stat(WORKDIR_REMOTE)
        except FileNotFoundError:
            sftp.mkdir(WORKDIR_REMOTE)
        with sftp.file(path, "w") as f:
            f.write(script_text)
        sftp.chmod(path, 0o755)
        sftp.close()
        cmd = rf"bash -lc '{_export_env(TEST_MODE)} cd {WORKDIR_REMOTE} && nohup ./start_workers.sh > start_workers.out 2>&1 &'"
        ssh.exec_command(cmd, timeout=10)
    finally:
        ssh.close()


def _start_workers(master_host_for_private_ip, mode="cps", include_master=True):
    master_priv = _get_private_ip(master_host_for_private_ip)
    if not master_priv:
        raise RuntimeError(
            "Could not resolve master private IP (override with MASTER_PRIVATE_IP_OVERRIDE in Cell 2)."
        )
    targets = []
    if include_master and MASTER:
        targets.append(MASTER)
    targets.extend(WORKERS or [])
    targets = list(dict.fromkeys([h for h in targets if h]))  # dedupe

    for host in targets:
        _ensure_workspace_and_locust(host)

        # HARD RESET STALE WORKERS BEFORE SPAWN
        cleanup_cmd = r"""bash -lc 'pkill -f "python3 -m locust .*--worker" || true; pkill -f "locust .*--worker" || true'"""
        ssh_exec(host, command=cleanup_cmd, timeout=20)

        if isinstance(WORKERS_PER_HOST, int):
            desired = WORKERS_PER_HOST
        elif isinstance(WORKERS_PER_HOST, str) and WORKERS_PER_HOST.lower() == "auto":
            nproc = _detect_nproc(host)
            desired = nproc - int(CPU_RESERVE)
        else:
            desired = 1
        n = max(int(MIN_WORKERS_PER_HOST), min(int(MAX_WORKERS_PER_HOST), int(desired)))
        bootstrap = f"""#!/usr/bin/env bash
set -euo pipefail
{_export_env(mode)}
cd {WORKDIR_REMOTE}
echo "Spawning {n} worker(s) to master {master_priv} at $(date -u)"
for i in $(seq 1 {n}); do
  nohup python3 -m locust -f locustfile.py --worker --master-host {master_priv} > locust-worker-$i.log 2>&1 &
done
echo "WORKERS_STARTED $(date -u)"
"""
        _write_and_run_bootstrap(host, bootstrap)
        time.sleep(3.0)
        started = _count_remote_workers(host)
        print(
            f"[{host}] master={master_priv} | requested {n} worker(s) | nproc={_detect_nproc(host)} | reserve={CPU_RESERVE} | started≈{started} | mode={mode}"
        )


def _run_master_non_blocking(mode, label, total_users, warmup_sec, hold_sec):
    if not TARGET_BACKEND:
        raise RuntimeError("TARGET_BACKEND not set (run Cell 8).")
    label_ts = f"{label}_{TS_UTC}"
    users = int(total_users)
    spawn_rate = max(1, int(round(users / max(1, warmup_sec))))
    csv_prefix = f"{RESULTS_DIR_REMOTE}/{label_ts}"
    html_path = f"{RESULTS_DIR_REMOTE}/{label_ts}.html"
    log_path = f"{RESULTS_DIR_REMOTE}/{label_ts}.log"

    ssh_exec(MASTER, command=f"bash -lc 'mkdir -p {RESULTS_DIR_REMOTE}'", timeout=10)

    try:
        collect_snapshots(GEN_IPS, [TARGET_BACKEND], label_ts, "pre")
    except Exception as e:
        print("Snapshot(pre) error:", e)

    _start_workers(MASTER, mode=mode, include_master=True)
    time.sleep(3)

    start_ts = datetime.now(timezone.utc)
    run_time_sec = int(warmup_sec) + int(hold_sec)

    start_master_script = f"""#!/usr/bin/env bash
set -euo pipefail
{_export_env(mode)}
cd {WORKDIR_REMOTE}
echo "Starting locust master at $(date -u)"
python3 -m locust -f locustfile.py --master --headless \
  --master-bind-host 0.0.0.0 \
  --host https://{TARGET_BACKEND} \
  --users {users} \
  --spawn-rate {spawn_rate} \
  --run-time {run_time_sec}s \
  --stop-timeout 30 \
  --only-summary \
  --csv {csv_prefix} --csv-full-history \
  --html {html_path} \
  > {log_path} 2>&1
"""
    remote_master_script = f"{WORKDIR_REMOTE}/start_master.sh"
    _upload_text_file(MASTER, remote_master_script, start_master_script, mode=0o755)

    master_cmd = rf"bash -lc 'cd {WORKDIR_REMOTE} && tmux has-session -t {TMUX_HEADLESS_MASTER} 2>/dev/null && tmux kill-session -t {TMUX_HEADLESS_MASTER} || true; tmux new-session -d -s {TMUX_HEADLESS_MASTER} \"{remote_master_script}\"'"
    ssh_exec_quick(MASTER, master_cmd)

    for _ in range(30):
        if _tmux_session_exists(MASTER, TMUX_HEADLESS_MASTER):
            break
        time.sleep(1)

    deadline = time.time() + run_time_sec + GRACE_SEC

    def _remote_file_exists(host, path):
        pkey = _load_pkey(
            SSH_PRIVATE_KEY_PATH, os.environ.get("SSH_KEY_PASSPHRASE", None)
        )
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
        try:
            sftp = ssh.open_sftp()
            try:
                sftp.stat(path)
                return True
            except FileNotFoundError:
                return False
            finally:
                sftp.close()
        finally:
            ssh.close()

    def _remote_line_count(host, path):
        out, err, rc = ssh_exec(
            host,
            command=rf"bash -lc 'wc -l < {path} 2>/dev/null || echo 0'",
            timeout=10,
        )
        try:
            return int((out or "0").strip())
        except Exception:
            return 0

    stats_csv = f"{csv_prefix}_stats.csv"
    while time.time() < deadline:
        running = _tmux_session_exists(MASTER, TMUX_HEADLESS_MASTER)
        has_csv = _remote_file_exists(MASTER, stats_csv)
        lines = _remote_line_count(MASTER, stats_csv) if has_csv else 0
        if (not running) and has_csv and lines > 1:
            break
        time.sleep(2)

    _stop_tmux_session(MASTER, TMUX_HEADLESS_MASTER)
    end_ts = datetime.now(timezone.utc)
    print(
        f"[MASTER] mode={mode} label={label_ts} users={users} spawn_rate={spawn_rate}/s | run_time={run_time_sec}s | absolute_window≈{run_time_sec+GRACE_SEC}s"
    )

    try:
        collect_snapshots(GEN_IPS, [TARGET_BACKEND], label_ts, "post")
    except Exception as e:
        print("Snapshot(post) error:", e)

    return start_ts, end_ts, label_ts


def _collect_results_to_local(master_ip, label_for_files, local_dir):
    import os

    os.makedirs(local_dir, exist_ok=True)
    pkey = _load_pkey(SSH_PRIVATE_KEY_PATH, os.environ.get("SSH_KEY_PASSPHRASE", None))
    transport = paramiko.Transport((master_ip, 22))
    transport.connect(username="opc", pkey=pkey)
    sftp = paramiko.SFTPClient.from_transport(transport)
    base = RESULTS_DIR_REMOTE
    files = [
        f"{label_for_files}_stats.csv",
        f"{label_for_files}_stats_history.csv",
        f"{label_for_files}_failures.csv",
        f"{label_for_files}_exceptions.csv",
        f"{label_for_files}.html",
        f"{label_for_files}.log",
    ]
    for fn in files:
        remote = f"{base}/{fn}"
        try:
            sftp.get(remote, os.path.join(local_dir, fn))
            print("Downloaded:", fn)
        except Exception as e:
            print("Skip missing:", fn, e)
    sftp.close()
    transport.close()


def run_cps_locust_once(total_cps: int, warmup_sec: int, hold_sec: int, label: str):
    if not MASTER:
        print("No generator hosts available.")
        return None
    for h in GEN_IPS:
        _ensure_workspace_and_locust(h)
    print(
        f"[CPS] {label} | total={total_cps} | warmup={warmup_sec}s | hold={hold_sec}s | users={total_cps} | spawn_rate≈{max(1,int(round(total_cps/max(1,warmup_sec))))}/s"
    )
    start_ts, end_ts, label_ts = _run_master_non_blocking(
        "cps", label, total_cps, warmup_sec, hold_sec
    )
    local_result_dir = os.path.join(OUTPUT_DIR, f"locust_{label_ts}")
    _collect_results_to_local(MASTER, label_ts, local_result_dir)
    return {
        "label": label_ts,
        "start_ts": start_ts,
        "end_ts": end_ts,
        "local_dir": local_result_dir,
    }


def run_throughput_locust_once(
    target_gbps: float, warmup_sec: int, hold_sec: int, label_prefix: str
):
    if not MASTER:
        print("No generator hosts available.")
        return None
    for h in GEN_IPS:
        _ensure_workspace_and_locust(h)
    bytes_per_req = max(1, int(TPUT_PAYLOAD_BYTES_PER_REQ))
    rps = int(round(float(target_gbps) * 1e9 / 8.0 / bytes_per_req))
    label = f"{label_prefix}_{str(target_gbps).replace('.','_')}gbps_{hold_sec}s"
    print(
        f"[THROUGHPUT] {label} | payload={TPUT_PAYLOAD_SIZE_LABEL} (~{bytes_per_req} bytes) | target_gbps={target_gbps} ⇒ rps≈{rps} | warmup={warmup_sec}s | hold={hold_sec}s | users={rps} | spawn_rate≈{max(1,int(round(rps/max(1,warmup_sec))))}/s"
    )
    start_ts, end_ts, label_ts = _run_master_non_blocking(
        "throughput", label, rps, warmup_sec, hold_sec
    )
    local_result_dir = os.path.join(OUTPUT_DIR, f"locust_{label_ts}")
    _collect_results_to_local(MASTER, label_ts, local_result_dir)
    return {
        "label": label_ts,
        "start_ts": start_ts,
        "end_ts": end_ts,
        "local_dir": local_result_dir,
    }


def inspect_state():
    if MASTER:
        out, err, rc = ssh_exec(
            MASTER, command="bash -lc 'tmux ls 2>/dev/null || true'"
        )
        print(f"[MASTER tmux]\n{out or '(no sessions)'}")
    for w in WORKERS or []:
        out, err, rc = ssh_exec(w, command="bash -lc 'tmux ls 2>/dev/null || true'")
        print(f"[{w} tmux]\n{out or '(no sessions)'}")


def stop_all_locust():
    if MASTER:
        _stop_tmux_session(MASTER, TMUX_HEADLESS_MASTER)
        _stop_tmux_session(MASTER, TMUX_UI_MASTER)
    hosts = WORKERS or []
    for h in hosts:
        _stop_tmux_session(h, "locust_workers")
    print(
        "stop_all_locust: tmux sessions stopped. Worker processes continue (nohup). Use pkill for a hard stop if required."
    )


print(
    "Cell 11 complete: Orchestration loaded (pre-kill workers; snapshot timeout raised)."
)
print("NEXT: Run Cell 12 to start the UI; verify workers in UI (should be > 0).")

In [ ]:
# Cell 12 — Objective: Start Locust UI master + workers (uses selected Test Mode + payload)

import time, requests


def start_ui_master_and_workers(mode=None):
    mode = (mode or TEST_MODE or "cps").lower()
    if not MASTER:
        print("No generator hosts available.")
        return None
    for h in GEN_IPS:
        _ensure_workspace_and_locust(h)

    # Create a start_ui_master.sh to avoid quoting issues
    start_ui_script = f"""#!/usr/bin/env bash
set -euo pipefail
{_export_env(mode)}
cd {WORKDIR_REMOTE}
echo "Starting locust UI master at $(date -u)"
python3 -m locust -f locustfile.py --master \
  --master-bind-host 0.0.0.0 \
  --web-host {UI_WEB_HOST} \
  --web-port {UI_WEB_PORT}
"""
    remote_ui_script = f"{WORKDIR_REMOTE}/start_ui_master.sh"
    _upload_text_file(MASTER, remote_ui_script, start_ui_script, mode=0o755)

    # Start master UI using tmux new-session (no send-keys)
    _stop_tmux_session(MASTER, TMUX_UI_MASTER)
    cmd_master = rf"bash -lc 'cd {WORKDIR_REMOTE} && tmux new-session -d -s {TMUX_UI_MASTER} \"{remote_ui_script}\"'"
    ssh_exec_quick(MASTER, cmd_master)
    print(
        f"[MASTER] UI start issued (tmux) with mode={mode} and target backend={TARGET_BACKEND}."
    )

    # Start workers on all generators INCLUDING master
    _start_workers(MASTER, mode=mode, include_master=True)

    time.sleep(3)
    print("\n[STATE] Master/workers after launch:")
    inspect_state()

    print("\nUI access via SSH tunnel:")
    print(
        f"  ssh -i {SSH_PRIVATE_KEY_PATH} -o IdentitiesOnly=yes -N -L 8089:localhost:{UI_WEB_PORT} opc@{MASTER}"
    )
    print("  Then open: http://localhost:8089")
    print("If 8089 is busy locally, use 8088:")
    print(
        f"  ssh -i {SSH_PRIVATE_KEY_PATH} -o IdentitiesOnly=yes -N -L 8088:localhost:{UI_WEB_PORT} opc@{MASTER}"
    )
    print("  Then open: http://localhost:8088")

    host_url = (
        f"https://{TARGET_BACKEND}" if TARGET_BACKEND else "(unknown — run Cell 8)"
    )
    print("\nUI form values (copy/paste):")
    print(f"  Host: {host_url}")
    print(f"  Health path: {HEALTH_ENDPOINT_PATH}")
    print(
        f"  Throughput path: {THROUGHPUT_ENDPOINT_PATH}  (payload={TPUT_PAYLOAD_SIZE_LABEL} ~{TPUT_PAYLOAD_SIZE_BYTES} bytes)"
    )
    print(f"  TLS verify (locustfile): {LOCUST_VERIFY_TLS}")
    print(
        f"  Timeouts (seconds): connect={LOCUST_CONNECT_TIMEOUT/1000.0:.2f} read={LOCUST_READ_TIMEOUT/1000.0:.2f}"
    )

    def _sr(users, warm):
        warm = max(1, int(warm))
        return max(1, int(round(users / warm)))

    if TEST_MODE == "cps":
        print("\nRecommended UI inputs per CPS tier (users, spawn_rate):")
        for tier in [10000, 25000, 35000, 50000, 100000]:
            w = int(CPS_WARMUPS.get(tier, 120))
            print(
                f"  Tier {tier}: users={tier}, spawn_rate={_sr(tier, w)}/s (warmup={w}s)"
            )
    else:
        print("\nThroughput mode: examples for selected payload:")
        bpr = max(1, int(TPUT_PAYLOAD_BYTES_PER_REQ))
        rps_1 = int(round(1e9 / 8 / bpr))
        rps_5 = rps_1 * 5
        rps_10 = rps_1 * 10
        print(
            f"  Payload={TPUT_PAYLOAD_SIZE_LABEL} (~{bpr} bytes) ⇒ RPS: 1Gbps≈{rps_1}, 5Gbps≈{rps_5}, 10Gbps≈{rps_10}"
        )
        print(
            "  Pick 'users' equal to desired RPS; spawn_rate ~ users / warmup_seconds"
        )


start_ui_master_and_workers(TEST_MODE)
print(
    "Cell 12 complete: UI master/workers launched (python3 -m locust; PATH exported; quoting-safe)."
)
print(
    "NEXT: Drive tests in the browser UI or via headless (Cell 16/17). Stop UI via Cell 15 if switching."
)

In [ ]:
# Cell 13 — Objective: Optional — Open/close NSG for UI port 8089 (if not using SSH tunnel)


def _get_nsg_by_name(compartment_id: str, name: str):
    net = oci.core.VirtualNetworkClient(
        oci.config.from_file(OCI_CONFIG_FILE, OCI_PROFILE)
    )
    nsgs = oci.pagination.list_call_get_all_results(
        net.list_network_security_groups,
        compartment_id=compartment_id,
        display_name=name,
    ).data
    return nsgs[0] if nsgs else None


def open_ui_port_8089():
    net = oci.core.VirtualNetworkClient(
        oci.config.from_file(OCI_CONFIG_FILE, OCI_PROFILE)
    )
    nsg = _get_nsg_by_name(COMPARTMENT_ID, "nsg-generators")
    if not nsg:
        print("nsg-generators not found.")
        return
    rule = oci.core.models.AddNetworkSecurityGroupSecurityRulesDetails(
        security_rules=[
            oci.core.models.AddSecurityRuleDetails(
                direction="INGRESS",
                protocol="6",
                source_type="CIDR_BLOCK",
                source=UI_ALLOWED_CIDR,
                tcp_options=oci.core.models.TcpOptions(
                    destination_port_range=oci.core.models.PortRange(
                        min=UI_WEB_PORT, max=UI_WEB_PORT
                    )
                ),
            )
        ]
    )
    net.add_network_security_group_security_rules(nsg.id, rule)
    print(
        f"NSG rule added: allow TCP/{UI_WEB_PORT} from {UI_ALLOWED_CIDR} on nsg-generators ({nsg.id})"
    )


def close_ui_port_8089():
    net = oci.core.VirtualNetworkClient(
        oci.config.from_file(OCI_CONFIG_FILE, OCI_PROFILE)
    )
    nsg = _get_nsg_by_name(COMPARTMENT_ID, "nsg-generators")
    if not nsg:
        print("nsg-generators not found.")
        return
    rules = net.list_network_security_group_security_rules(nsg.id).data
    to_del = []
    for r in rules:
        if (
            r.direction == "INGRESS"
            and r.protocol == "6"
            and r.tcp_options
            and r.tcp_options.destination_port_range
            and r.tcp_options.destination_port_range.min == UI_WEB_PORT
            and r.tcp_options.destination_port_range.max == UI_WEB_PORT
        ):
            to_del.append(r.id)
    if not to_del:
        print("No matching UI rules found.")
        return
    net.remove_network_security_group_security_rules(
        nsg.id,
        oci.core.models.RemoveNetworkSecurityGroupSecurityRulesDetails(
            security_rule_ids=to_del
        ),
    )
    print("Removed UI ingress rules:", to_del)


print("Cell 13 complete: Optional NSG helpers ready.")
print(
    "NEXT (optional): Call open_ui_port_8089() to expose the UI publicly; close_ui_port_8089() when done."
)

In [ ]:
# Cell 14 — Objective: UI API helpers (start/stop via API; preview stats)

import requests, json


def ui_base_url():
    return f"http://{MASTER}:{UI_WEB_PORT}"


def ui_swarm(users: int, spawn_rate: float, host: str = None):
    url = ui_base_url() + "/swarm"
    data = {"user_count": users, "spawn_rate": float(spawn_rate)}
    if host:
        data["host"] = host
    try:
        r = requests.post(url, data=data, timeout=5)
        print("ui_swarm status:", r.status_code, r.text[:200])
    except Exception as e:
        print("ui_swarm error:", e)


def ui_stop():
    url = ui_base_url() + "/stop"
    try:
        r = requests.get(url, timeout=5)
        print("ui_stop status:", r.status_code, r.text[:200])
    except Exception as e:
        print("ui_stop error:", e)


def ui_stats_preview():
    url = ui_base_url() + "/stats/requests"
    try:
        r = requests.get(url, timeout=5)
        print("ui_stats preview:", r.status_code)
        print((r.text or "")[:400])
    except Exception as e:
        print("ui_stats error:", e)


print("Cell 14 complete: UI API helpers loaded.")
print("NEXT: Use in browser or call ui_swarm(...) / ui_stop() as needed.")

In [ ]:
# Cell 15 — Objective: Stop Locust UI and workers (hardened)


def stop_ui_master_and_workers():
    # Try UI API stop (best-effort)
    try:
        ui_stop()
    except Exception:
        pass
    # Stop master UI tmux session
    _stop_tmux_session(MASTER, "locust_ui_master")
    print("UI master tmux session stopped.")


def stop_all_workers_hard():
    # Kill nohup-launched worker processes on all hosts (master + workers)
    hosts = ([MASTER] if MASTER else []) + (WORKERS or [])
    cmd = r"""bash -lc 'pkill -f "python3 -m locust .*--worker" || true; pkill -f "locust .*--worker" || true'"""
    for h in hosts:
        out, err, rc = ssh_exec(h, command=cmd, timeout=10)
        print(f"[{h}] stop workers issued (rc={rc})")
    print("All worker processes signaled to stop.")


print(
    "Cell 15 complete: stop_ui_master_and_workers() and stop_all_workers_hard() are available."
)
print(
    "NEXT: If switching to headless flows, run stop_ui_master_and_workers(); to reset workers, run stop_all_workers_hard(). Then proceed to Cell 16/17."
)

In [ ]:
# Cell 16 — Objective: Warm-up run based on Test Mode (CPS or Throughput, payload-aware)

import glob, os, pandas as pd

# Ensure a clean slate for both tmux (masters) AND nohup’ed workers
stop_all_locust()
try:
    stop_all_workers_hard()
except Exception as e:
    print("stop_all_workers_hard error:", e)

if TEST_MODE == "cps":
    warm = int(CPS_WARMUPS.get(1000, 60))
    warmup = run_cps_locust_once(
        1000, warmup_sec=warm, hold_sec=60, label="warmup_cps_1k_60s"
    )
    print("Warm-up CPS done:", warmup)
else:
    try:
        targets = [
            float(x.strip())
            for x in (TPUT_TARGETS_GBPS_TEXT or "1").split(",")
            if x.strip()
        ]
    except Exception:
        targets = [1.0]
    gbps = targets[0] if targets else 1.0
    warmup = run_throughput_locust_once(
        target_gbps=gbps,
        warmup_sec=TPUT_WARMUP_SEC,
        hold_sec=TPUT_HOLD_SEC,
        label_prefix="tput_warmup",
    )
    print("Warm-up Throughput done:", warmup)

print("Cell 16 complete: Warm-up executed based on Test Mode.")
print(
    "NEXT: Run Cell 17 to execute the full suite for the selected mode, then Cell 18 for metrics."
)

In [ ]:
# Cell 17 — Objective: Full Suite based on Test Mode (CPS tiers or Throughput targets)

SUITE_RESULTS = []

if TEST_MODE == "cps":
    for tier in [10000, 25000, 35000, 50000, 100000]:
        w = int(CPS_WARMUPS.get(tier, 120))
        h = int(CPS_HOLDS.get(tier, 600))
        base_label = f"cps_{tier}_{h}s"
        res = run_cps_locust_once(tier, warmup_sec=w, hold_sec=h, label=base_label)
        SUITE_RESULTS.append(res)
    print("CPS suite complete.")
else:
    try:
        targets = [
            float(x.strip())
            for x in (TPUT_TARGETS_GBPS_TEXT or "").split(",")
            if x.strip()
        ]
    except Exception:
        targets = [1.0, 5.0, 10.0]
    for gbps in targets or [1.0, 5.0, 10.0]:
        res = run_throughput_locust_once(
            target_gbps=float(gbps),
            warmup_sec=TPUT_WARMUP_SEC,
            hold_sec=TPUT_HOLD_SEC,
            label_prefix="tput",
        )
        SUITE_RESULTS.append(res)
    print("Throughput suite complete.")

print("Cell 17 complete: Suite finished based on Test Mode.")
print(
    "NEXT: Run Cell 18 to compute CPS/Throughput metrics from Locust artifacts and collect snapshots."
)

In [ ]:
# Cell 18 — Objective: Compute CPS/Throughput metrics (from Locust stats_history) and collate snapshots

import os, glob, json
import pandas as pd
from datetime import datetime, timezone


def _iso_utc(dt):
    return dt.replace(tzinfo=timezone.utc).isoformat().replace("+00:00", "Z")


metrics_root = os.path.join(OUTPUT_DIR, f"direct_metrics_{TS_UTC}")
os.makedirs(metrics_root, exist_ok=True)


def _find_stats_history_csv(local_dir):
    paths = glob.glob(os.path.join(local_dir, "*_stats_history.csv"))
    return paths[0] if paths else None


def _col(df, names):
    for n in names:
        if n in df.columns:
            return n
    lowmap = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lowmap:
            return lowmap[n.lower()]
    return None


index_summary = {
    "mode": TEST_MODE,
    "generated_at_utc": _iso_utc(datetime.now(timezone.utc)),
    "runs": [],
}
any_written = False

for item in SUITE_RESULTS or []:
    if not item:
        continue
    label = item.get("label", "(no-label)")
    local_dir = item.get("local_dir")
    run_dir = os.path.join(metrics_root, label)
    os.makedirs(run_dir, exist_ok=True)

    hist_csv = _find_stats_history_csv(local_dir or "")
    run_entry = {
        "label": label,
        "paths": {"stats_history_csv": hist_csv} if hist_csv else {},
        "status": "ok",
    }
    metrics = {}
    if hist_csv and os.path.exists(hist_csv):
        try:
            df = pd.read_csv(hist_csv)
            name_col = _col(df, ["Name", "name"])
            rps_col = _col(df, ["Requests/s", "Requests/s ", "Requests/s."])
            # Use Aggregated row only
            if name_col:
                df_agg = df[df[name_col].astype(str).str.lower().eq("aggregated")]
            else:
                df_agg = df
            if rps_col and not df_agg.empty:
                rps_series = pd.to_numeric(df_agg[rps_col], errors="coerce").fillna(0.0)
                avg_cps = float(rps_series.mean())
                peak_cps = float(rps_series.max())
                metrics["avg_cps"] = round(avg_cps, 6)
                metrics["peak_cps"] = round(peak_cps, 6)
            # Success ratio if available
            ok_col = _col(df, ["Requests", "Request Count", "Num Requests"])
            fail_col = _col(df, ["Failures", "Fail Count", "Number of Failures"])
            if name_col and ok_col and fail_col:
                last = df[df[name_col].astype(str).str.lower().eq("aggregated")].tail(1)
                if not last.empty:
                    total = float(
                        pd.to_numeric(last[ok_col], errors="coerce").fillna(0).iloc[-1]
                    )
                    fails = float(
                        pd.to_numeric(last[fail_col], errors="coerce")
                        .fillna(0)
                        .iloc[-1]
                    )
                    if total > 0:
                        metrics["success_rate"] = round(
                            100.0 * (total - fails) / total, 6
                        )
        except Exception as e:
            run_entry["status"] = "parse-error"
            run_entry["message"] = str(e)
    else:
        run_entry["status"] = "no-stats-history"
        run_entry["message"] = "stats_history csv not found"

    # Persist per-run summary
    summary_path = os.path.join(run_dir, "summary.json")
    with open(summary_path, "w") as f:
        json.dump({"label": label, "metrics": metrics}, f, indent=2)
    run_entry["paths"] = run_entry.get("paths", {})
    run_entry["paths"]["summary_json"] = summary_path
    run_entry["metrics"] = metrics
    any_written = True
    print(f"[{label}] metrics:", metrics)
    index_summary["runs"].append(run_entry)

index_path = os.path.join(metrics_root, "index.json")
with open(index_path, "w") as f:
    json.dump(index_summary, f, indent=2)

if any_written:
    print(f"\nCell 18 complete: Wrote direct metrics artifacts to: {metrics_root}")
    print(f"Index: {index_path}")
else:
    print("\nCell 18 complete: No metrics written.")
print(
    "NEXT: Run Cell 19 for concise local CSV/HTML summaries; Cell 20 to teardown (optional)."
)

In [ ]:
# Cell 19 — Objective: Local artifact summary, optional reports

import glob, os, pandas as pd


def summarize_locust_csv(local_dir):
    paths = glob.glob(os.path.join(local_dir, "*_stats.csv"))
    if not paths:
        print(local_dir, "No *_stats.csv found")
        return
    for p in paths:
        try:
            df = pd.read_csv(p)
            if "Name" in df.columns:
                rowmask = df["Name"].astype(str).str.lower().eq("aggregated")
                row = df[rowmask] if rowmask.any() else df.tail(1)
            else:
                row = df.tail(1)
            print("\nFile:", os.path.basename(p))
            print(row.to_string(index=False))
        except Exception as e:
            print("Could not parse", p, e)


for item in SUITE_RESULTS or []:
    if not item:
        continue
    print("\n===", item["label"], "===")
    summarize_locust_csv(item["local_dir"])
    htmls = glob.glob(os.path.join(item["local_dir"], "*.html"))
    if htmls:
        print("Report HTML(s):")
        for h in htmls:
            print(" -", h)

print("Cell 19 complete: Local CSV/HTML summaries printed.")
print("NEXT: If you want to tear everything down, run Cell 20.")

In [ ]:
# Cell 20 — Objective: Teardown (guarded)

TEARDOWN_CONFIRM = True  # Set True to allow destroy

if TEARDOWN_CONFIRM:
    !terraform destroy -auto-approve -var-file=terraform.tfvars
    print("Cell 20 complete: All Terraform resources destroyed.")
else:
    print("Teardown guard is False. Set TEARDOWN_CONFIRM=True to destroy resources.")
    print("Cell 20 complete: No teardown executed.")
